# The base schedule is a plateau, not a lever — and here is the agent

I swapped the base schedule under my routing agent and my offline gate went from **52/90 to 86/90** on towns it had never seen. I wrote it up as the biggest single lever I had found.

Then I tried six more base schedules, unrelated to each other and to the first. Every one of them landed on **86/90 exactly**.

So it is not a lever. It is a ceiling that the *routes* impose, and my old base was simply sitting under it. That reframes what the work actually is, and it is a cheaper thing to check than to discover the way I did.

The whole agent is in this notebook, runnable, with its sources credited.

## The measurement

Same eight-group route table throughout. Only the base schedule changes. Gate is three opponents I actually met on the ladder near my own rating, 30 towns each, none of them used to choose anything.

| base schedule | record |
|---|---|
| what I had been shipping | **52/90** |
| a top-2 team's replay | 86/90 |
| Thomas Tschinkel's router, route 2 | 86/90 |
| ...route 3 | 86/90 |
| ...route 4 | 86/90 |
| three more from unrelated teams | 86/90, 86/90, 86/90 |
| Tschinkel route 0 | 19/90 |
| **Tschinkel route 1** | **0/90** |

Seven schedules, one number. The base only has to clear a bar; past that the routes decide everything.

The two failures at the bottom are the more useful rows. Route 1 is a **wool economy**: eleven sheep where the others run eight or nine cows. My routed schedules are dairy programs, and at the switch turn they inherit a farm full of sheep and no cows. They are not written for that farm, so the season collapses. Route 0 is a milder version of the same mismatch.

## Why my old base was under the bar

Comparing the first six days, the schedule I had been shipping against the ones that clear it:

| | hires in 6 days | strawberry seeds bought |
|---|---|---|
| mine | 26 | 4 |
| the others | 27 | 8 |

One hire and four strawberry seeds. Strawberry is the most expensive seed in the game and an ongoing crop, so four extra plants compound for the rest of the season. That is the whole gap between 52/90 and 86/90.

I could not see it for three days because my offline gate was two opponents I already beat 28/30 and 30/30. A gate you have already won cannot report an improvement; it can only report a regression. I wrote that failure up separately: [A gate you already beat cannot measure anything](https://www.kaggle.com/code/zhincez/a-gate-you-already-beat-cannot-measure-anything).

## What the agent is

A recorded schedule replayed turn by turn, with four live layers on top.

**Routing by shop group.** At day 3, and again at day 6 when the second shop opens, it switches to whichever schedule suits the town. All eight groups are covered. Every route was chosen on towns held out from the check that confirmed it — four of my route candidates ranked first on the selection towns and then landed *below* no-routing at all on held-out towns, so this part is not optional.

**Hand alignment and weed repair.** The recorded actions assume a crew size and a clean field; both get patched against what is actually there.

**A budget guard.** If cash plus this turn's sales cannot fund the next 72-turn block's purchases, surplus stock is sold, highest price first, sales placed before buys.

**Sale pre-emption and ordering.** When the opponent's public farm looks like a copy of mine, sales due next turn are pulled forward one turn, with the quantity repaid so total volume is unchanged. Sell orders inside a turn are ranked by how much each loses if the opponent's visible ripe yield is quoted ahead of it, rather than by headline revenue.

Depth beyond one turn on the pre-emption loses: lookahead 2 costs 5 games on my gate, lookahead 3 costs 22. The reason is in the forum already — advancing a sale is only free while it does not break the chain that funds the next purchase.

## Sources

Every recorded schedule here comes from public competition replays, which are CC0. The agent ships four:

- base and two routed schedules distilled from public replays of **Milan Leonard**, **Mengfei Li** and team 16730357
- the wool specialist used in yarn towns, from a public replay of team 16686918
- the routing idea, and the five schedules I tested as bases, from **Thomas Tschinkel's** [replay routing notebook](https://www.kaggle.com/code/thomastschinkel/kaggriculture-95-5-win-rate-via-replay-routing) (Apache-2.0)
- the sell-ordering key from **lucifer19's** Harvest Nocturne and **ahmedberatozer's** `_r37` layer (Apache-2.0)

My own contribution is the measurement: the gate built from ladder opponents, the per-shop-group loss breakdown, and the held-out protocol that rejected four of my own candidates.

In [ ]:
# Builds main.py for submission. The agent is embedded below, zlib + base85.
import base64, hashlib, pathlib, zlib

EXPECTED_SHA256 = "07f6c80605271b028b3aebc803180fd8b1bf3162a2d394bb30fe65b8d943d1f5"

AGENT_B85 = """c-ri}>00Vaw=nwOtBCCa1s08<h^VNjD2)QP^O-_`01-$q2_SaQcM<0b&y}1Sqbf-y1a+==?fo_9d!N16l%!JasL?k|OG_7Gr=!WuQQsJ8l3<8KDL7DdQEm!iM^cQCa5$3thR~P_4f03p?}6BpYP3aYI1Mc=EiEi83u*E{p)G2IzniL}8=_+9<Sjm4AWgDLM^+@kdMC71O>mCv>ye**DJUspSyPpP1kEC$STwZd4=pU5N{Xb3hSU<;nmP~$!<y0+mG0l<Kb@MchMsg)NeL}gP*g*ZS~AX<yljOQaEeFe(q%<R3vYiNWy<AJ^{*ZBXIs{Eqc)YLeyi3Jr{q~Qupn^%`%6{~NgIoO@+$0lH4tCur{vw{ymzeInCrc&bW~aCkT)k{Uzc2ObxDFwqu~$o;;%wBUn<U-Zp1mM`O|gGHZkXw*{O3AzMQN%*KIo2O0}H1J<OKN_dlEP%&emxufN!6@A-R;VK1VpjU<l-Y$1^_T}ic6$W%*nR<S)FN+SOOlx)w1lG!<%^NrK#-kxPMKZEIi2Szsk97H1FkB^0gOfgr;<k?Lu-Qtv+s%}7^VSYZ}Z2LT*!I^ev*YMX+Q(L1ZjPq9^m%sRnf2SK7c}1`gyuHj+s@LW0U%0QQrBWr!PcjyEZQATKH|OqmWOh@#UZBBGHhX&dmuo-y*&dZjmvpL?vuyT~pKAR3sV3%5H8yvuiMdm~EtT?L&DGjj0tc7n(((0C6#(HEgWiB(;^u&L{{jx25-_o0fbENUmgd;PUnkjeHJ8slWXpg3URWq($`{#c?J`r&6lSo0E==!x*p1oY^=91tRBWhYsn#ATO<0TmtQmiKwlGAkBf)-zLtCx}4N-@Uh{fH{rM^0`y6%#jB>m;t#OgZ|qW2zlVeVRRA+ea_dN^0nS37^rzU5Zn_?AuIu`}Pz-97PeXcH&SozqumVJ#!-`Np*x!efZqUvt~>lkjN7mKX6H$+^t=Bh6W^O~mJ{l%AFQp&RD@@_KCz77MxVCR<U|1<UTmIDx=x5mCAbxNGy*fJ=x(9pvWw6LK#i;#`DvDhcOU&Lzy;#@Qt#ycm>k!7lP|f}Lw=t_4QF2Rjd><Gt7|HxKhAuA8}A{{+`1ym4J3{2${w^A6cO&R2J6^Xoe_3+!LwJpK}Wb%$oRXm%CbbQM6TzPm)>_!pPRg$8D^K-B93xeoao9N>il2-U8fnNrkY<|2!i=$A=}>GAxAH6E6)i$`a*$|X`-R|_Xe$Mz?gav@u;)F`>ePkzc0;P<(sZ0)F=%@k@dLHypuHJDt2J;zJJ7aV0yve=ppLSN(_9x`@E=BM`PTql)6sZ>47WxpI3Ti$2N#ag9WD$@ZG3VxFv&1Z_owNm*wTSiW(n8o$o;(xQi!eE2TGFc+>N&qkcI85?i0_tJXqFTBw){e5pYPMYaFJY@}Mk*I39V5&F2M61Jk}s8D>XC(ovt02sTRY9<isbipC=6fS;m*`@#Y#0(q)XUduoMPVjqo>BG;r#I>Wmg*VWBJy`{Gm=L_w39s@9TPsH_kObE$x=kN-|&qbnF)O_GGZJSP7Ur;?`c5Wo(iGC&afatCz_JER#^cl@QZS4~4!6*7P(2}7|(2KbvWuSye@4u*>Dp`k8JBxpPm`+ZO`jHMLzKOtfcncGm5wH-3cks=%9vp#-%Nlvl)_u+Lee=Mj9d5eu(az_`CDHmV@8e1U@L?<IC(u;vBqeC=CEpi=&&Pdc+f^XUtr`r)SeO-PD5`LaO4+vY)Al;O{+U)869YJmjO;PU(!~O_*mv!utd`QMq6fzNA>`OY{{w8^#8UjHJEu05V06T)_Bdysb*hCKq`bfjx^e%jEkoN+jA^gZ}uTB(H`HTibSPc$92G_}^1ca{K=}KhfLrrc<bQc1+4bTHlh@M1WY?2`wQd`wn@1v<6S^y9cbg3`lM!|-+q_#Nf8x$%~Nrbx)hE7S(hEUfDcr?2p`qYK4I?{;zOM8TQ(9cv2o-An6SRzv(OY5pbGLZoha4mo#0MJ!+Ng(|MlVs*ZECN7NM}{yE$p8{P!sbZR2%wSkmINR22p!2!wNdh5sA`4)IxF*U@@3cjTh-x)$O?Sa+VN1U;Z{Q%80P}@W<p;TTaqS7FH&=4uqiaiF~Ta^GHC#J1Ljz(!OvO^I_~#{hDgqif%in!dW0Yoz$ACfP_^m8Y3YbuS(S`R<`*K*6>%W>=KU5MI{fQ1hw%IT0DMd4?<KM*q0?6zVqf?91*uPVk$g*Fx;bh2{0oPf;wd-WlPW7dGJ}Ay)DjJGiTv9TmCh1rm@ge&kd|9f#4(6JD~Os1iOL=m#C}N;l}*u>1&Jyl>j1My;!teL#<ZRiYLl)c8Z|>B`$z7e*b+c_Na2nAH4$}zK!XPG1?RDlJI-f;1TMk3NZJxX=SxJgtt|Z}^FssykKo5Z)OwPEzdjbOi#dWntJy-0l2G3gA_Z8~Nj7^-|79`}{2*_SvSg_X3uIt&f(@VI-@z*+280HzE<lhj{05+{!Om)Ab0n)7*R(<4U)T^D_*|3lUWlaUTDEYQJwDDJ!+JjWy&~rYfCG5df}<UJN8rST;GXc>mm3?vTx`b$wT<o0E2SbfVzUOAtif*|e8lEO00O%p6Sl^;XUwhu^qbgpKZPN!XKWpCi9#@t?t|B%7NHJ<p(g3Nk9HDjY{%hYOY&Lc{el_%_WMKZh2OtG*WDsV-B*VLdU5;H9d7-CG-yaI3Jp`b7X&^C#uO-GA-9MiMwMb%He&@Ev%U#2(E<Cj3+p3r$50w0bp#}P1W5=jmE2#{W539{L8nHw-c;>QmZ^zk3TzB(m+_YIXY73AbKEPqYXLk7zyH?~G+O#?0viGBnnEa)?t|xKQG^d9*jZ9q><kGKyX_K`%F$5Z;Vc?Tu|A&hFYe=jK&Y>(0g90gN*^qmLP$LHjI^VsNzWsKgQyWekqn(5>G1mA<Z}dI1T=!6JEE-UbUqx#2@p&zlY=4Wp%HRp;0R>f6}~%qO9>(Gx1nJ}g6UCOEVV*HMgVvk0TaMlB2z(IG!fAPF`yHnEHuJ^G^K|R$G!t=Fl0(Y281<$wuWe;Y1rzJP*Mn67fl4i$zXV!JSW@Rm)eGa=n3dom~1EZj}#IJF9D_&a10583XskTk+?-e33Rpa<i@~iz+NK^MYhNr5xJ6R$UIDw_7k#91_ERdApdV+7sJLb>@)`if+V1;eMvzeA*h~1JERME_{;bW*Im|SBwkI)$DRfViD?>)0sum_K84{%2sa|o1)qs!{l5tU>yzu?W6vq3^^=V!=#?AWL^0(0jCN*3Wda=J97dA6`RG^pH(@^ohLmv=VQC}3uqQ-j`yt!(YxFDfn|uU+<CNfcKnVNEL?ez5q&`0aVtbF7iOR1f<k**>Q}UCplRo-Uzb5sV)48p3bB@40|G_YnE`7T}&twEyflZ12Mkc^&9Hplo@tmZ&r{?6iJ$>den+dB7W6ffX8O#+xv=*TJrVjikp%FDhB^Ss>YL^TfnVh!8k3u(ae<`cWVGo-@Hj$nlGA|VCBu%8}FhvaHqSpxXL-3#>4d5^5T?0f=gGOY2_%}4uoo#U5Al{MlSQirb?0F=%G_<Rxai3wO^lV28OpWwpHDOm~sd&5e+yDQU8?Fw?d;x^roxqEq-+J@>EHE`6IAJyD0o?Y{S|1O@7oYvu(M3QAI>++Q-bgm}Iq1X$FdnCv&t?$I+zBK6@m*MR#y9Sc$h2q1{c7g(5tiMJ3u`&o;6`2FEyUHC*J@l{I+nzLK7U>Rbr{l`i3viE-QeKw`Kw)+fj@io9Q--Q@;k2Y0In~)rA)|S*K}w0+6~;-?wO5E1Q=NV0M>79z+mnM*c*J7D|^8VV#Z&9x8ec8XOL|`ROT1{{#%G{0(wb%Q8v~oaFw8I^8&^;hhYhb3A;E|PBZ79{fHv~WL&Q8FGpX$JwLRYd^_g~+<yeN0XZl*7<6@+osc~lZ14-3|NgZEcxDMm$@6@4H}FDt12<`&H$%p&J11$Q4gOa7znV6s+|IGRrfos)!0VR$c-quc!8vQ@z3|m6Src=6!4^Mfd;f5raPbJULO27MiKGx#1014!2=?5@QH5ffXK(<5;UBWJFuht~$9<N)wFl0HfeM25y&=eco*AQU0(00GlSRWuHPli1sGJxO)COrvIW@!P!ZNLvP|4==rgR5W^iVijf*i+#5djXRs~~7FiBx5>V!&g7W~o7l0Q{{l{r-hNP|VYoe$#tMK+J4K{v4C-U9w&@k?q2F4CnpER^&NrgmXhUCD*D8*NS5U147Nk)XimDNn6-tRC2l5R=}7xd&iwIQY)s%VEaaxMC7LkPPMT%?z|W3{>_9yy-Pl9_f?S&L{_UR(v)Wwu_HPLr?_b&rpN08hv&RFrd`4bTsRm3Ex_5ZFxWH)@cSJkV%-Y`u$>1*NZN$fAogr(o9N430{c5|<q_Qt+EA%c?S7h<d}U<}(<kH7EhE3lL8+Q7>9ZscKIjDm{DL$L;X(V{oGTmWR}Zf|`-Q#t*nNj3kNu`hj#K*e+ny5X)q6{af&)oHLu-fFUJMECin<QL%`gu&)sUJ7!ZOh5Fb}tmp2<c#7GOTNhcqn=gm#PuKI<&qVDuIAq$y&w`xM8b+-J;_1u@ppWKQ-1g60JHJ)qz3@!()>bx0fziPYos4XKP4psoGfX@qnFPQIQrP4~q?qa_N>9l;lDlGz7@CYdm6ixwm@OLT~{4$s*wYykG@xJF*^ig-km#NL8~iz4?1J0kp=FG%-8WgO3tI|V{1I<Cold6&Pr&+J5IsUVy!I~aP*8y;lOnhZEK&Jt3`BAJ*+Kbe^yFky7+glS9*wk6=FQXzh89{Df2O!I)AIi8oENH9${o!}rF5ZMu13jyGQXZH9uCLVkb^B@3!(QeUdN0IypFdj(RjGX$-$2Nfif`y&@+Oc=%H&44s^98WwW(&FN0-7ak!!R&sXc)#na`_AR7c9a{e&)w%@^-b-37SN*r>9Fh!V-E-S|5=IN2OcqTk;cK6_-BH5RBF>I$tntsS^d2qPoQFxXdKSF0vL9s-Rj0)1W2|B{a^W9=#zo#gR_+>H4TKkaZnU`?9ILC!9g4m_<|Aj;AX-Gq~wq_BK=lksbzWYef2j-vlnfd4&uB3>7N=MP-W5Xj?Q~GabFo43cDS$cdPme&`Y=O1=>d&=d=XJdkW7K?qr=n%*^SC;{OmAXFJic`*@mh@fanv0vD&H;pU+Thu>kX@>Z{%(T>>Jz5cv_iCvbo^98Ka40@I1+_EV8cd!c^2Qelg$d-p_`N=hgy(-43D5r&9gDb6M`W*FgujKVyR#Abp7UH-w|8^|@>vLkjCCQp;Mh3I(msy}ov9(kW%S2n4>*<LHTVHgBR3pBGeY90l(VhFr*$}gW?!&6A+O9o>lP;qMBd@n`U}2ioTdnnm?Lwq0xR%q2_Ar5lKhzqvp=c~e}LM~8flk&#cB5jL-sxA1%y%zbVzoQL8R3J+=!t2DgdX|O$RtK_+SHjT%Y?{E{vQ6n~c>5Tr7`C@%~QUld}D#zL->SW>U`2?9B<%Ep^B|oB~5r35M}?fcd6y0g^+%2Bzd`4ctI<AIdq%#W>(X2g4~le*g~r5G{%=jYy)Tb844ofB|yO4r@4+1a#yU#<DIq`iL{g8<`p#8!C~q5LL<=6-#7lQIx5~MCJpI-c$&)OqSqWqc<f*)MPb)%|$|bMIw5kc;XPIm@H&#a4mkAX}ZylWVK*omc@=T>Wv@zojfVqU@D=CiabCq`3$|WxIWsUp&HmOY_VFJI)UZl5Wgaz4I*>;O{ZfJDlmtES5X#LC!YFdPS33}4foG&Oe!6lE{P0I&=gx9@D21Y>JmX4HVF*SX5QQuUGj#-qysFnPpOYT!X7UdvaQ7r9XPD7>N-PV&&D*s-^B*FwAf&Kq{@^)n?Pw0A_{O!0GWJRN53Z?u2D*_V=r|E;Is*la2&lwRTj(uT(yzpMC|u;0?P)tE}<vaUzQD)$hEql_f?GZ!heA%Gqg%A^gve0#Xv_am>)Ry;4X|51x?V_1^3mhc(VRTG-}PRO341s%c$0$!jHNiZwU?~D=q0oXh}`EMLmJY6|oG|DrF9tI|Rl0&5<^iLPA+o(6<J{(g<b%qAY-$5djox6_==QR!T4?sz7xFfC(DtmV<6>;1@M84gB=WNms_Z-=CtN5gm9SbVYQO!`;TM2bZA{_@fDZ*e8N4WW#Cw1tq63MX>{}1^vqWg)%3@9}eO(du^Gii~Xcr3CMn55KBTIo=|K3WfM9hXzHml#moC-E3(L9Y>Ye{HTrV1rjv~pAsPh{kIR>VhcoKcA>oFKh(dO#?_YpZ&DzF67UKe~nR5s^6$@R0hlxvu=Wx?6hK&9T&=48c6mFJ6ONfF{hycA3D2)->ECDw^qX#vy6pA6=^)1fyz|(EdZ{C>6jR*rn^md}5LGT+e))Is@SwC*k#3hh5d?0#bNRC{m))Kjy<hc{SIZdEWGh*PU22yCcv_ySnaOx&3HT&eI)mpN701Z?53Y_OGq<x`s^qbxsgKs|s`sMYpsmE&QzJUZnVi5Zye}V8FBkT=(6|r7L@D*Ii2tpeAh>>G{m50zU`a3||F{_jaAk#1xud8hj9j<+oN`Q%&VwUg|^fEAKQ`6H&kY1YL{_5?MimsnK;M&<YmA<xel`4LH9=@Ygr6j$OS@oyPLx!MDaB&PUcT(m`N{P*%D~5lp|8=U59jf5}L@}%Cyeq+fX7V?k2)@xKf35NT2X;JO<?&<V6|VdTW-CNoj_!YK!ZC;V-gq-Rh-u#P?(sXL)og!ggk;>zpBNp@5r1fEoMU2i&d0;jY7PVDAZmJAm@&fIUH<QFPQbPP6Jx@^s<!{1qeGw7n-4+B8pTd66*~Unxx*jXb^i}mwg1o2sXOO8%YXe}W%<9TEN}KjT^9<dH>Q(EC2dV6f@ZDeBb-Y+APhxK9O&sHL3V!URmVxt*FcuvKDezTpN7mh7J?L*vI2aw$$i?dNyA8Zb89makH>1D%?M-KlO<ONG(5=q9DkDU@pkx42Y(1t!nhiR2vxCgBp8cCcUWW*kq$Rw@xK$%%>W9{n~5a+0H=TQ(>WLoc(O;MzsYcL&i2HXi7;<ST1{^Cr5d?QwH9Fv!}K9YQd0pQ0Maq20iQe*pfMHSeZbIBeOE(O2**gkurITiCEC9Msb7>J%S0i7@`ONxq7yM)_~motuh@wA|Nc7~5F8JG`%{F-vI(I;jk9qK0b=4r9g$fOSy9p^vd-eq$lwssr3=g_UIH1LDgj8>A?KFaI{~UlRqgxO-q|m#<MPk@)Y?Rf8fJPh-;<sqe*PQ?d2Wu#3^hXhTfk=nI%m;pyh}ep7~+_aa-1iu54t8T=^1fb0UO{v2f7u8s({Zu4eA0_4#w3$w2=yWO)~%hq+dR~i^iB>52QUh_J`achbwUI9ep@&-{{*K95@JW&P{W@Bz<ho{a))3RO9_MymIX}eoJ@Sb}O)6^yTM~+#=gr!(cQ=ug-U*+i29?uN}>1-#+`<lAlN3%K{*x9Tt&5TRD4cc2x*tuzI4uoxRw-HHlayHu{ne<KqHWteX`e<OgJ7{o=s`J`1*pbR>a(V+3B+b}S}9k4b%KFsG_8!wGdgg7H96-BSQ|e0ce`qW%;&0w11Z26}*kd@#cAMFwa5l>airFt?&s2n!P>U2s4qEY%v*8cn5wv$0X%>`N+naR<LFlz_xwFbSx$R1iq+1xLJLK+r@$FL22eXn~syL=E&pOo6XR!cf&^n5)=F%_KKQ-I@w~S`&>r2Da%T;X)c3W~5}0AoCql!QT=h(Z9n<#?N7prD@^}38l6WOC}wce4>zJkHjJ{xd}lFc6>A2bwU*Ao>DU|4WtYPh0P>;?!!s80A&IU$NVrhH^$~R*>eI6Ex{-D(U>AQzeofuqpy3tbh!j~fr*r~{rm42NI-rxiEv}YHYp~<MIZ*03H>uc1gStr98wfU$i6(VVtCjbl0v>z$yRpAR&4$q4TstJ@N^FdP>0-PD@Z94+rp;6x&V^3`vlDrL>wXT(pw6##~2C(J;(|H3)G!2@S%D@7EN@j!zW;)R#bs>*OA~D3_UaljtS*8q4<|E8j>6RLIsN#WaizFZ8o=bWWRaHR_HMcDq#_=FgZMZh*CloI-WEGck7M`xNHWrFa!eZh}s#r0aW+Z5pi44Ei#4bTJ#1~VXrY585T0>PT5IWVRge9lFSbcm>Mu~&%fifCu47qUBSx<%(;KGdlM!2P4}=5ULf(!MKj<7>3MW`Z)QjW!I6ma7n}~gZn_cUwzq>R_WH1b!H)2@6k8(yg%N1r=Lcay6ElxOq0oo$|Nf8v$Gi~7EV9~E!3`O!Ao7gV6QE)X=lQ5f<(S=3VJJ6ycEDeXN9dq60~H>1g(ZVIkvcboCwh-=t{+<n-e*U4Y67`6bWmCM&$%ilYT>|R{KaY8Uwm=m+<{0f+PO};a7_UtIH_lYLqwQu1<F7x=9X{j>Jel(Ks(N`U!KDIkR!yC2_TP#6#aQ5ji^zCZDLEpFiEg#iNbO2ls^H2x=s2-V;C{SdI#wPy7|xmSRKw6Xc$rmXvbPx7Bt;}AV*O85)|3zh9EMK&Mer3U{RYM<-ZAcflf8_X7p*qDdj#{lYy#Xlg0=j17l1@fpT%&2tH~l2P`~+sW9@}2gw!EJP`Uk9Flvcfc<Dh*j*tS2q)ug2>>12u~4i>Cj!YJqJd~K%42WRBpD%1{6J>gu#xP@2;eKgxU_^J@B)C`$;&zyt&TiCZd=_gVsIhtP62`Ynllg2{n{R4em_<dFV5M~kZ{=aL`{vLS8rw`=q>S8cNqGb7AT7L=n3AFPJG*hd*1ec)ssU@>CqFsCq2c&;W>N7&K|6#U;S^HnZ8-`zNpje9svp)mKlVD4{+TIqREL|21%lE&4TYHvgv0=lA#DGBieSZ5U{mq(m);!m>0dmq*T%s$Ivd@4)i7rqm>5CP}D5aS5<-&`*M%8Ah#r7W+?&kV%X!kPz#-<OFUywJ>;`xrh_2zMsLR!ET~=yx<|wVeZlyIWgtfa?47Qn*sXWvHu$N4iN{(gFAcz&A;Sln>-ojv@&E(i0!3=q6MQ(Ezd;-&r*71&+%!612zdsP(sk8f_GHOGa=jav_Cn`1a=#_}lW^?KEsbACZvcuAEf^S(V;GM5NUtZRyN{Ixr43ZrWMoZnAhRQ<*ccfOrG`XVNHd|%Kn1aX0P?u4!4-V5_}1SYl<WWsP{}L!qRA~k6;o`lQPMOP@dvme-~*T`Vr1t@;rYk}zbTT8GLH_VNvEoOh2X|6xfnXL`vrqGTaHd!Ja5<*$h<W|jVQy>HPdzw;btrY0_QxL&|@^@eNbj}h6AlQUQ&9_TxRFM1HhJuarU0)ZN2tV7?_O+CiMQAyg+O;_d~aif_veGqs`Y7LhgwiG-vJfEg)1#RVO62!!tM)FQe3&%T~CMA=(Yd<)%mxQ6OH|08OLRGO6t>rdbxsh=xRjE0)2?Y>sfnGPhWHnpnG&a;X4DAi_q9sG@TZ5#lc&C?M>UjL_aEs&%ix9VbOyd+laXt3#TG2@xhjT@$W&-zmM}#3$t5$}K9Jz>TJ%qo{sEZ$AmxBXXgeEYAdRM99D~=RG<T+po$)W}@ti>9WEctPD+Nc*P8_;uC40T?mm1**Op-3FcPeDh{1XphyTdp{c3NnVE%SHNZ7e1UE7}*$J}8aF<ap2wN<3T1WD{2|2wboj(cRX@B(EQpbcq)nUgNFxFa9!_eyilWKwJ*8ogxCenz6L-Q^uU6Lo2VlU?@6LHX=!8oRr88MI~kp6}?9@9r|0*o_ZNqO~<Nrx~KiD6)(k*3;FWSw%^6H`ZPkujk06^ewxw+s~-x|x;=DEJPK?FsZ!_f69a&VnA?{-J6fK0IR1cFeRwjht<m(=6GiUNJ=Iv5YwU#=73l+gaKS+t|p6!(+4W0iA7s)lFA6^!c51DF+;hdUt`Q>RMY9n{MvV{T|rpP&n?O-4)nEA~ESmu0_2kSw?}%S>`g!qahS3AotJ?cSNc}=_DLXLsWV$5f_h09FQw!m}rb1B;zQV7LG^{6cOQUh2w0#nyHYB4tg1olPnc0^Bp!dd<JM}AibT?(kNONfz>oI0-38|fD;Z?wNQ;cb%Mk%fj1*^8{TkmZxPzkghv_@!Db2(azxWllX7Y;LaQ~xbCD^bQ4_5IW(<f1<FVZWUWB<nqM1eN8x9>qPm+dIQ-mr^Utdy(nmSfB#3HyCLHr29#TwEVI3$unR0Rwb5yH43!g0`;_9Y#8QMi~Q+nA=PL5QG`Os<;q9gCWb3?TH?wYX8_V?zEpmx)er&RxQnUtc9`^2Jr+--D51^Q&&=T_yXii+!Ex!BG3&7s+}(dyi&1q1?^fPAHEP36pNzy|DXAu>6?u>R%#mAH7I$?)YJVT`2tF?wStXS39=X2g3p&qaoIIq>aQJh(Pd>Sr`o(k_qCF+!7FT7ftOn_arno^#fermZku7pJe-nwb9U0dXjsC`44oKCQ68GQj$yGfe{6a0Vs4T1YXq%PjN5BrkREjEP9q@5OX?;=y9lO4T=m*G68%!^4yR;P8UYQ9ZUJ5iG+ah*VFdwZBW-`lQm&JeIoS0xPt(9&v63O=Nz?&kzkaO)1k!UFXSyjak7c{P!9<O)6l_iB`@^s2=Q&asd(6OzemVQ6d%A%R++PJY<txyakM4Pko#n18WqJcs)Znd=MmX=@{nD9(1*h&;L<P|5exW&myuvomC;dJ)%XKA98HBjcx`KDR#S;eMP3Hl`Kv?ub%&-2i-ZX>;|Yk2gd-V9Bx1oxED((*geLm;<A{++kbn#@MAV^~ZUPYoOSWa6DFv8{RuTtX_tX{4B1@4d83Xck5>WJkK0%0XB?a&>Gn1!*=Qdb}L17!++V)d{kL(k>A5C@OfFUcU4i&&HbD!zG!k2`b5{Dt-4BfGSYj2WGgQjSj1~F1)Ipa_v-6XRLxovfH1UO#C+AxY{iW(uI;vE;D6aEc$jq09J*#bW1JP~Bx=GeQnQ3$%T1ReJInCxgvQdn+P3s(?sPEyUex;X?kYr;)VBr*k{n>w?c#<t=+0>o&V26zb2L}wa>z$s<*WIJdG1ayDMA%tP)m>7%7?aX8tz)hmEJ9|WJXq2@F;}p3giXjuw`ctX?2Lk=Fa71(V@qsFs;E&2`kQx~C@9``cP5DQY|FHza6z+72<$x#Sp&PoRnrYR}C%A*`iYOE06-`<`Ns92f6~L6ufT+MVg&>(1N48i1{UEPBTE^={PFm~_06LkijKTp!43JD0oW(H;7Vjb1RU$s39L8Nrw#Sgrl)pdb$<S8>&9~(j2Szt@A~NAk0+*$h+$LyG@I|*MYXivD&j<^&5?y-~BwPLpdPce$J1sMIjFBxgHYJh_Grc;ILS$veW=ZEnXgKP$bwe|$KlcLMq^Npk$?X-zS^)sm5cnOOjQg#ZS{yuIJb5YZg{M5ykw>22+0V&mGhYoxgd@^MYSYF4S<gL{smB0ZfKPNgO{AIBnNZ~rxw8oF<m4nn9~q(|+ge|X_w_}hF<}g`RJ<N<G;ATHS!pU0{s7B`%-q1}!8Vb2lZ&u~h^;&A$b^wuf+Wqj^As=Og*POgp=MwQ-VBVeh|IB%=76DTUnH~47YZjtx&qYPY>tM&aib}@>LTyFK6n6EaC@7OHeeiJPtZGrZkdSaF-6mNP5_)+vJMg2aKAtr(gwip^TFnP-<W$@wfG-}Rw2WG(EwyH9|$WeDLF?oXiGRM<%$(-*A_7fkco=<Qq^@67~AnGc{!0eU(OQ`h&Wm6b>&(|7HdsWb@kzT`&B38bxrAvR5T_!m~!V^WbBjkdN5SE))Jk!!}*SUKO+@%Rnmma(KUPX@w{o{d+#}OuW-UdnxQ5N0*7Qn1Q={DEwGQ*33Mo30y4;7dkoSfS5;S?%i>-zWMNTjN)!R5<3I#GkP9Vy^y{@<+zTp(*;tsoA=q6elOzivSEnmWr00<eVU&QGWohV=X5ipEkYbYwH$=RHP8^9SIpItk3~TfPxkXj$&Ah_^4{z&R4yoJEy>a?7UUELdaS6i*m&{NdppU3rWh%`bCcT;Q(sl@SK)`u2Whc*H0#A<D6fh|ZjHWSDG{V|;9^yD3%BT<7Z3oe&JL)ieg#3qN1ko@gOAn{97*P!TUb48thTQ3Z<^vfA<R+I~7=cDk(tU8L@3~iW7ttQN>3I(5WaK<T>-oF`M1x!nczq{4+Rwdcsr{DHT<VSD`MK4=i%4?KzzpqW$n|KlL~Ccv5O^Z<*5K%z)SW%0%it9RPz-}m(v&TL+G6JnLcnf-9DCSx_*byXFhe^uF4%%7%U+9A9f^#`7~cEoc;$5>HEcn@n>Az7w`ii7b!26V2Pfdrhx4F@zMV5y*Go|6Y|iNbyafi-Ti$W#uGgU3n$-t!jR@cAIS2O6I{Rf*eDp!IXo6!PNGF=5%!8riVPEndh5(<Z_k;UAOm^mUsJ`0NFMVfoV5%!BV6^-=O;f!D;u7UL7wq5#>Vw2$7cg4|rWyoaF25sUhPo=JS>z9TnkgPWbmmJI73B7Fa=E23{}POe^jSG0K-FXWPB~*6slSQog}C5$z@u)orVsGU^nQWAzh|CL>R+swbw3nl6JJcbJh8m<eLC9&9DZ3ewV&;sg(;Qi_rh&fhyb7u(o}o4rlBd{u4U(okLLi9>2-_A0<7A)STw}wYT&{GG9%Qo5qp^Do&)?JKnmwl_h-=ZqwV>3VajqOhH$UgOqyY`+Au_sK{FbriG6tl<?~*A?rlJ*#5qxiZ_?Xuzt_QHCzyi@{`ePQlRj}(>-Y(@0hD<=o(jxG)0`dG0=+s2Y%I=%-OcHp#U_|Nv+P^*J@UO9nEtX~4*Bna4LfNEDV<#qOmo&xmxmJC)(o_8o}&MYkiRd{;N0!_IqG0@bcomULlPZ@U$}F$@$9_Q{NWw4Kj3aq__rhZ<saRfKff;)HPs|=;)M@53;@*yBrPZnWgRqFsD7lH3YMS_bJ9_Pp}<O?nd#%0<J}07CxAHwbInoJzSTcfl+%hM5<08V@GP_eft8iUQkdhrLMkvMOH@=EcLF9!GeN?Cp%?PEj?cWC@(+Wmco(F3ZX4fO1%GDR^K}3Gl54XeL~I&1uf+k0#dI-d6jMn63;dgfuHyp?3FAQ9V6<tS$aTU7e$RuwfmGN7x+$ExIZeh2x|l#a!1-qhGt43p7ErKC5r#=1&F55i%w@FUznEV|f{I*bA`Mq137YPL7UjaS60}v9hH-S!S>LnBggtjNxC+Ohi3?cozpXkI^K#7CG=L@NusZY4%pd>#wWHuH%{l)z(-1<k?;tG<0PXz#8B)Ru4mu8(8tc-|^g=Vl;fti-G(UtivaUM2h`3mv956Ku=CXdtzj<FkN8WtWi#GvTwu~lb?M15cWIoV9CWa%Fbiqdok)CO=Ag}r=Q|aIiF3by4qy`ouoFc)YqYRk?6SFE)JGs)b>^O8wL3J$mr8(*O$*RmyczS~JZ*PXld`otcel=8wzOZ*&mfh)#J#%A5>}}qG1Q3~d59Jx+Z&ylTlXh2bu*)!DrU$El9Tv8%Va_J&mzmLNPHBXffPitU+>AZjf%X(3_}BFJub1Bzt373S1m95{g3n+0=j0oPI~-`;wH%LZTGZi(Pcwyy2X4J<t$3ZxFE|w+tkUm?!~bRO>HP394pe?E9p_G6cXk;|pQ4$C+Fa1C0N&c<{!GxQt_fpNk#!XL$sK_tl5o2?BPU^#-~z#(pt;U50@~&Xft&|WvuNXE2CMWWl=FV|=r4=!G)Xzx@17~=!Fa#Ds_eJ`yvMuSn265~b>DGqQ-<yg-6RKoA$tM{mS(!7f%~R21!l}+JkdmxK`RToy^wI5sb<Tr5tvGt!zok|?684@p+`}%W(Ig9m_9`K*O*$ZnG-DOLwAi}nv_kq7aEOaPtkM$7FhU`K5FQOY+yJwt4au)3~>b`t%3eK3*N|=hDBl=Wk|@_8H~w#3Z<Luv6&nIxyc=7RHk8<xHW&DWy&|%N>z}Rc_YC3U_~$sWKh|%M1yusJ)O|e2<>=WbmqTC&Y*;udB3PFPV;eS&L0|)IcPpVCJsR=yz^&>^Ct+6A$Z@PVKDz6&cuZ>7ycuZ$5qCHsjFO`3BtJ3Tw!x2Xs0}dWZGaFCHK8HQrO3|V$)uFUS^A(pCL*FLJl5G{NS{M&fo|>y9jrgK%GUWC-9j>0_X>Z&LOtXSAb^fRYwg5&8{&pQ^sEe4U2Mr%CeV$EsDCcPE_iYMQ}j9FAj(lBG;NwnOJNQ91`V`b_6y5uHNg`0wK{W`3rR`dngIisqvhp1!X>0W)gK87{ONQa2)fvzfLDPvR6l?d_D^W#$($_-v5nS&T&{PYDgCnk$~j_$lJ=TR3uo~lKOvaMws)NBsRN@HzT-*>Pe`h2r3=Pnj+ibkdTM4Ej;}IOSt|+G-~NJna=<s&{>?x75Ss+j>WwGDRPpPu|AU(aBek`|K;!+U<&{%KuNF^oF|B)_zQ;RwPLR7X?~B~TK{KZQk=z~qEr8$p8<@62@v@Vs5C=#z}|j_v;OG-e}dh=1Q<ILwHKBmpb?Dt{R~<CVfPN?Z<i4K?I9UMVjgGjrqoR}9khjy`LSB?W|myy)K$ze$t!QC&$AAme|A0&|1|;%WO-dmSPhE`y4}#6v$tl!^j5Sn;B>03;q`0ZrXuCzkqKtwQ!ue{tjG=IlzpObOyG~;d^6C)@=YZe5`vKYX3&Wv-47yA>PQX_xKu;Q94^ed^E9_Bc&9R_XsWbi9#9Siy`2n^)8ko(L1+Gxq<dVB((e;Xasyk%+SqS*tu`x9AAkO=_F9yt*FpWBdE(AQPdgqY7^B3gYnjxtR3q3)G`eFm)!+<@zHl<<B!E7=5FM<WN|VrnS46re1mJm^uK!cY0#RF?hIu_qO$~E5R8<$<YpR18?Ja6KK`v!)WJ=@$v?Q!u)sE`Bki6lLR{dcB!lhZcQx2H5y6irFNOHZYD$-qZrrtQ}NA<p3-S)nQGOrCZD}~Wt^D2coHe+tCQA^Z^%Hng=e{MKO<H~*UP(PEX)RAqH^l#<|#k+<O1*IOo$~<XfxjBg@q2D>0HuHjFITY6gwN|i!J<Q-{(`C?o|IWbSg@vqJ-Z#{V1N1A>kdP4Y4b>#VB7*kBj+KFe8gLBM<AN})Z3&GX?)z8L#HJm)Y{&6J5OffSYMEHE0sJ!MMUglN8<BVb{8%a9Oj&SO_r6FZlw9?P1k1Szkbuw-TbLGsy0OZb6HL-T3^>f^ipTl;6kP&7X1==S2DnBrtJdhuwQh>V=BWfivHcJ*Se<iFtW&pfKrbCtZJCG*gEO4_L%ln{O1k|c8o<*5D>JBo`L~b^><TgXg;ankr@p?UbJ>%=j4pT%H_DJM8O#F4NX0J?7=|xRsHgrx<a1~G;yG7M@xix1%-pW0-jm0XJoM1$%}Q``ZwIXpW4{B#IhVw69ZX>C#KxNFw3Ge!AC4xP`#A8Y<{#Ctrl3duoXt*=23;8wwkloj6@E5*63wn^ojdcFolPw5S~3n`Vo5rk-B};yb=r=`&!~<T^g4av@L^8HVq2hx4Uf{&JW$W`Dr=yFGW8r*M63-1-pZqepA8*e`0Aj%IbBVhb)K-Dioax!xptI2wwRhOXk;maG3&`<`5k}Rx!sKKd$v2yJJxh^GC4mF57#e;wA(wE<8Xa4$2GD2mpwuwudVUy(G?Tmrj!^pBx{<AQI94)4X{uLh=`I}JEm{8%*sQc$_Qx-dX@Ry1<+=+Ii#i7Y>oyapn`S<7A`2%5f3^O1xiH}5q;IXRCc$ST(x3)c+>O?l-f^3Osb%2NYE%34_tK!<9mUZ&E8eZ87@%O<PuyXP)Y|`oiG-R`D0>kGj#8;m;fvU(nlc-v|~6f1kPZv9R|XHS@b`lmfx_V-%+yOpOUfOe+LcwA^YpKHmjhRhbE3wlEAlr$P^%I5191CxhodejrpkQF3hDBW<4ect#F0mFml1QivHnrTcaOT2KZV8FjMBYMggx#A8%iQf1)h>mb3V$dH27{zn}8(Zu9oSbHTVN1$~o%|Jzl6D<?gB+{_;eNY68<pZWK@37E@G-;2})_~I3K+2}`JREM>kOPRTZd|y&EuHBczc=$G1)RX1``kHvsHeU{5e8pdIX<N-08#hs{b9_dfC0fC}xSXnn%Lr$a!(nhVG|d`zS_TL&(&rdySx7k!mEtSef)SmCsZ&tH%qomfVc@%gHRaIHASC3;GTbDOtbyi$P*`5q2_a+<a%q@WF6s!0)dzHzMS&7Z#FZ>^3+5-bXim|YWl%*Tfknj#*zuJusV_HB8MI@H@u<pza1ZGPYY0Nv^9Xa`n+2+v0?oY&M`|6J#-fvt8&(xV>rh0)?#s4$bMaleICYtk{*b+7UI$QenR%tLq*7QKg}I}dk0Lwpj4=c|tS?863qAqW667dYveGYO3aKE?Y(}d?SY_|XhS0=9^tquDMAEe@G%NZ9;-D~GoOe`RwsIn&@v|{%k?&|tF*6F+m`NaJCThSuZCg>MKVbC{#daIx_4!z3yHFYH6tx%Cvs`7i9vKTE9vN7G+03)SoHH<88WcrBuyh2sLni8$M00mbFU|f)m&X!wRO4z;+qnp7TA&uDfe6i@23EHt&5mG3i){$2!Xeb|^YG_%00qCLe3>@RiP?dh93RiA@a8NK<(e;i<C^b3?vENo=CAZ0uhLni$qs$u%_v;_li~E;hIA}uy7CEMCxbu|0woLZpP(X(!1jnajrXa`3`M8FAh0^8@V~0*Awx4xFs~w(Z4Et&MlxU)C|qtOrJxb&G+PDvX}NM?xeOgJtjc@z_PxXAeJli?N@kAmcR!zhkpCDzBzOUm5pipfb(U`5&Q*wHP^T49K(%3teH7XA`}$A(ib;1fwk`hU_m%qS-bhf-lUd5hYg$xJ$1>UY%V{iYii26J6SFyx_H)e(vtb!OvKf%!sN$de;idDpKB%D|O0A1d^ssX^%*E#8Zvpu~sQKv1V&QkBa=|>;G*VC$Drco~m2gLz{0i5qL7<<#fuaI5%L4txj9sQQ9U8Km0Jw3%ZOWRW6iaUG4$VQOkD&5IIN?aJ!zZC+S1gvGO0u?0{Toex6O~UN5e)~nqhV7UZdO}}ZwKOuSV*|Vl-$^f?Y6=w2y@`a{NPZGh)<#CNT-P*JAC?K+04B+2yu|2)Cg1tgxZfz#7+ako$4}XMhXaKML?XmUDJ@})^jADp^OdV3h_39vjI(pM6ZD(b68W>(VIocONL<{B0V5oex<#T#bad&GD~VUpx%G1R&|3GR*1v}Ukqy+^Jy$+k1(?3=M4_jG{80<u~`$HB00vWgSk)mcmOX`yUU+#Zur12{wj|kg-X-Ypbw*hz{q68@NequjHM6*T#3ejqsFBF_kug=TMM;Ps+qYEt{1h6So@1rR?MxUIpSnYBS!6KnUiX^%vf-=pmQUjbEz?tuN9NbX_8&;8Pj)#gj4H~MV8M~m}n}hl0=%`-eI<<C#>(VJob>0g8X0DVOpkH<#Z^+5}}uqK!V9JA>>GU6TENj18qP`2|^r+up^_9ATWbZzf|2oMhq&7vn%FQPPpn+&Q)S8%qTlZ;~@0z>H#Ne$;$(*1RQ{Nem8GMda?BQv!xtCWRA?Dc(NUH#P`<dH-`%bBbQ0V+$jhT0m}@L7NnMsQ2~>r1O%>dVsE@71bbtsndpiY!DJw5j5ERA0!;PAcFAq_$(|PrN@i-W$SUw^t_nZ=?f=0t#(%mDzyW_+g>tTjh2Ikf1a3RHNt8VmV1KwrW&zoG+q5j0^^V74f5a+b*|m?%JW3|@Y}1Tw%xwq#%c<lGVy<rC9Y^7wFM6P&xmn|!pnXUpZ_ybbF%(gbbPK0D$Uc#&nwT|;V65SYVj;?H(!6+0O`QN+3;2qoZw^GFcQG@dneT7bzbLYW1PBmbBxLb2RBgbrNf{vjLCy?2og(H{gFU4gphTA9(8aVMlID;w?xO>z&6Ft}5GeI7{A-t=b*6gMQT${%ejU5bat@;Ho%}zphI&#FP?r>-pCMb2K#oeMxfWf#1akvs8_UvL;qIH%ze$tkp(|F4laJwOIA5%t*~*A*OjV-_NsB26a~Z#xh$CVwR&*<19WHVyX3T067)+j8SdMdM!n2)OJ2Tn+UpuS_wXM)#F>~ZYKl{_9_<jrr1cbU_aRFZhQ8!W2WTAL7{o9>7qql1f%#{T60-BLuOt~LH>c`EF*2{)x$fAVjED(GA%}trE!7bWqqO)?TE#vaGP)^46vWPmQ-bz=K2~46#Hr5g`9rjQvG%SwA%4>^CwE*uVCb0~nfHpYM_fk`zq^4ojbeFk&b@QA6X|ec;88@Q4{LliBNxCVyLM%N!+d?$8C%s5en-aB`RG-N*>|=<vS)GEG7ie30FB?XI=wM|yo!c0Op-p{(NZ(oM;b6<Wh2R+(^WSup`*+>szIT!PBlozUyT*Md7(4GSII}poz@F*i=KOrdAZqWM=D~e^>uvfUJF?96jhM@}m^L?y#vrzFI<+kP(B}4?D~PTAC-hL0SWPx7TO(qICYI|#zg0I<wcSx%6C}9tdBJ-dehpB$O<VHzEerec`HXA@`V78vqPT)OmaG)NOzk!h(e#h}0dHBb5<24|L2~3U+7hH31Qv(hc9UnrkQ0{FnxX|TAGAHQbMmS}p3kv(6e=F+*jQ?%grz>FDzr!KiyEeiV@WgVfJ=rfDNBx7fgb}oUJeVf?0F0in{huqiE<3oc4O9`u+ABaUs^IQB;zB@7)?XRlzr9!K4#2nOlvq)-5;Wi4<_oTteWLUY5L-u<tEJIvAJH#)7p2^<ZTDui1o)>K}x#$!s4b=%0F=<|Lwv?bNV&00Z{n`m%9OZ=9RI=t<v$ZFTzX#CS9fNf6lLZDd}DPQ~3MY2p+ue+<p;a|MA4V8SS})QPB7DXhPYFX)Ptwwr4B5OP80WVzyWXpjWX3*P%*e3%oo!RdPuz@i@(}>m8nkI&)T%hq(xZEFUNujiUd;)GWV4G#n`u`W>Q`%PH~4qF!9;W#*YH(`q~rr|d3U-ZTrnP`xeX>jz>Fikd+38Cv*^+1W(J?TRJ6G$m?umTBe~Rwo<<#uJgcM4>vBAqURJOi>3+z%DWDl)t@#KN;9__E8g)P!p-CTXK>(I!_^MPIqp<uIt2}0yfxyUfra{8k?go)(o61v@Pq>1ovDWALOh1DyH>mJ8<Z1mQujX)X;2Ofx9d~R3;4VB`q$1x<x=mR9ApP9{AQ-Wu77BsNjpo{|1dZ(;mhX<QL3+FyL}&B;(CkSW5VrD>#w@omud$vNm=#DTtFK%rC+EL&_TTRmjC<N(R&(%xQOh{ZUhHOIjrP7h4&mKO`)tG5x9-OUTdTb#hH=gITn0-5w6JV>Z`-K=XULv!=T6KT}2FiMr4Y{fHcz?sHo35%+Ms7fy{sOOY_791S=>0+k}<kp$_GO%Oq#h9Ro-b~6_MP<r`!|G6(<l=Bt!5cj~p(*dIy>exYRD{>VXsM-Oo;j&q1f>SjR-o{Wz5YzzP6<t<3+$^(fgK4V;VG@|o1@gP2ng&8q%|upoT>PG51zM>Hg-F&JvP2j_D0HxNyp8J&0+b<YhGaml+ch1a>?5nI{SS?e<YH2XJsRPQQ(OA&H=XZ1vdjL2TvKmx;2!3&1eWyyC=?nW_>52BazMF#*DSo$Ae8H!0US0O<{hE~;7n?Ez%f&Nz-WFyuTMbF;kk;08ThJd@_<t!PzMR*WdT0}iL9Fr%XKEU(;^hm6~|&9Tkbr7Q1B|7i3voj^7#&}32N0gGYg_a)VqQ17j0lMC$w^d)i=d3+re0b>P*4;2x}Fnra}X1*g&tWa)pd#G|-*RPk^o87l(kz89RcZL!>o-WURa}A@(#iylCq^kkKIUjq_`fb7cMu>`rkBM{M8X=13dc95Cq`G=Jm-nZ_1m3+hrq#ToRr5ecW<rv`e#M=+9~`q}7i8f_{6Y<y)oRY8e4MEguRLXJUfLM0FMszy6u`0wbZHzUrXA$L6z7XJyGYiBE7^avt$HVJs7FviDkj?(M;XyBtN1iS_`<^{9{NJ({{-%eKHNZ9<7Y5(t{LTHJO-~imM`B5wm?Wi+@c2ICT`Fd+7JQNb1MibYZEd4=Cba<@(hgt*XoH`e=xm;8oIIav8t|*eDIfrKTrLzUVK=!2>dsws}z3)0JMQZhL7ws^qFS(?W#x83^>WX6-!iUWQK?E^D*_TLSJF*pzY_rNrv3Nj;1`^3wXxq<6Ca?-EX5}cLjaV=SDg<&hhOo*K;-EJ9EolS`fwN^8IMA&~lF16JJ`8&H@!2V{avE1whPK1JK=C;JmZH08gsNG<;KDjX?8t1-SJ#<hHCMf7x)|G?2ZJU+PwT@fMsbU~#h(d(NT{eRkeL++;t)BSmZ<L%Jt~@hgo75~4B#J0#7gn58bKkUG?ug=VMSPeI<r&=Q%XA(z_dPyONB98E<_#-p(Ya45r7=?EqegwoJE5tIivhMT1o4>+1ir{VT%-&Dhj+4nHLz<lS7z}1dbCfVmF*A{0X=e>xP9*WxI#lM<5=fegP(p<m$W;3To5c0Fn*A%eBn*Hf9|X)(NPe+IHO-d0<wVq83q=ea4E1fkQNHYtNsS$0PDk-Q~+(=PSI-UN~yGjBHW|2x`C@i`sM!$@2_^|G%33e<9%g$7;R*9)jyhQz3T8Lq)agT^!(toz8l<SV}v0^u8zQW&#BIy?AA^nDu8M?Y}k^xHD{VbrR@>*_mIy4l?4=x{0(V96^KN^NOgDp>uaGL??1LMM;7L4AC&1uBm%I^`sq$Q2naP@vY?6qI))Z0i~E{k}aCm989gh3+20T8arc`B1=?Uu|r;kC!BFM)?+jY!?*VL>z>6gzG1Y;sBnB#$pBO0)D$j@&<~TG_@0@!kW0hL#Y;VVOPdxC#dBSdSlZ5rOxnmU@zI?p7H%`bMqRF>wHOAd8}$HORm?4pDF>)-%BrtClh)diL~1rMF=9u&^Iv)wdh-A7wC}QqCp)jvboe)Vr7^a?#Mll`w@614V)ypr%U?2-E}$Nh@E2P$zB9cf&GGE>!K8fJNa4;WKpD0gVF`d-rU}`0oY)!q|1w8f#$A!~nW<fB9U_boKvrkS)kX#R&Wr$Yxy@q%Pr}CxNiTJ(nxX6eU1u@As815j!kjWe1_KqiAe#Ntic7KiEz#y|SbA~Jsfm_UL}j;Zc++{s5#!#KjGEHH(l$N=%0@VL3}W!%Iy)ZnqBbbgNh5_Ek=c_g4#EzpGw%<60Jq1Dy+yt%17wKviH;Q6KTQ-j8}+CN8)+_^00TB&70HrpQP`QEYRjoJM|GsgE<IWnAT$T_k|B7oM26G>#bwzboSO=n2#L6bsz!tq4tmhrt&O#5$+brPpkN06P{k&5c(%*y+8#>`FvLEpk0d*5w1bUj1P*L9&Wx#aLC(CX-q}%srA5fJvJ`S`x0#a_y@Xq`j?U99wiAR%X``VrQ!rg`YVvRn=zg8W@QXC&-()C!Hi4Ax8+DnSmHB5Ojd~c+L)U>)%Wcq%Svh#vIlxOsP*SD|1<!B-4r=q$gD+F%8!b6+CdQ<BF~KVuF^KKIM8_f=b(gv}U=R@;86gVbIA{<F#j}bA+%zj+-9Nc-mg<A;9|XM1|0u6FpND*24stX9_<uN$d0g;+z8#)D1CW@g&>~fa7aD|?grEOmzoQp|uf9w~iTRk)v$@6LQcUejVW+0_dy7&p9&Nu!SNUC5%Kw7Ql-~xU+qr&S$s^&Mzs~}^>*XH*qk%!0D>iiB3*l!*FTk9_uPOYR&dsvlqIIdDM3I?rf|4m@sUZO@yo*ACPaxDL+itZ1Ek5~>z|%k9E(9LFSQ6O;ct-BR{7xCf`C0+Yk~#z7e5XGI^BSSPq%Q*?9X@wugd$SD#6mSZayoIgB7>d4?657__IJ-_DphjfQ>0h>;cXTle%1R3dN;o7opt2woelZr+!?%l-Fu~UT|JxWzE~>b(=TTa{Ttu)Z}#Nu-yC&@uC`=qx?>5j!992^E&K`2_a?m@o@d{>**ol)7ZCJX6HKgMgwIx?(6L3m371crE9fH7gPPzy3<<-SbI5B+1KK1R2`=XHq3!^vnuZWOB;2-q)gpb#O|g&ipKKYBQ3arhP^AmH)rBVbS^)Q_GUq<EO;x#eO+>=1qrQHmwPeXIqhm;*zYxh48cxa16NaS2@)#lK1ZC+}P$lRl-`Pb*7rb)p`Jlm$MVqtQ?|!#HiW`Y2h$8KVjyaV1v=r7`PjM1<+9PKc^xBxxT1Ix}30B*Pp5Dy4ebK8`Ud`&6dJK4ej1@p#Q59q`NgGpD4GoLO$Vv+`Xm@l6>F%K`fR!HUNp0+45_L4+JL5pnObT2e0TabeW80P{pu@()CI^qE8dmTMBW2o#TfsfBYju9%D!_sg(J{d3HpjGV9z8?|LilMf2(^sNkr@T4J>G%}dIGeFRhvLNs~r>xg#!X#BF<Hj%EU0hD&YuUmmKwtNN?0K3V+!N0qs_(rrslBYmlE9eNQjfj8+0{t`UkY=XW}=4X=EcT_+OaA^E3x$c7&bZEsNTEDOfOfjUwQM`})-wc3VA92R@y5-h30gT}t-L=J7Jttl5CK>!j$9>@#o($a(xb}%6X`l{#<QEV_3$g8BOyqyjse#U_9sQtst(qd-&XAa9a1Flq(GhUy^H~(wEH><qnF2?p%l{Nrc$ghAmVZotkC+|6~3Vt)yKEP$<2lxpnMSvPCK_tc-huwhs6IaZBN4Z=A;B`k$;c!c#3x5DFEbYftxC>f4L7GDEKvujQ8LR?y{GBz&q4)sC0E_@Lh=)vbJm0sW_CbA2hIT7`J*wQzz3p}@@7qq_|KV+SbUJg}>08%r=aW2rZ?5;MWz2GV5F$(~Yon%+%KKxqjk4?=mZOfsVg@yMnX+us^i#+{s5@&W{%e&)8zaoVl;-yD#ja6Iw(Nz5ydGvQvgLb>OD|s+kIrh9%MymbUgjPiGPTMXEzgxZ%GQp`*-XKDa*`<*vgJyxkSSkei415D-6ujL=B#sP-~x4}&cigD<nC-sFm2JPGIhGIuL7u2H=wTUx2$d7CrA6b=Hh(+n;+W=@qr^REN8p3f{Me=`|PZAJUho3kCtT$S_oG(>|fN)R0x>v9d(ugK5otJKdocFODhh?sehtA3@!s9KYN$ek9*~5N}5?x!1DTmk-YH9mxnap&*xuA=f@WFXBqwGOWc2$=+AQBC+w{OJ7h)nTbuk<Za&t?ZF}Pm$T3?CX)wVr8nyEe(9Dk>_ZAc!9)@Naxe1@Ot$bX%*Y3Ix%KwdHz3^2a87&gVs`6mCbsJUpW?-k&zKFuWf5f)n>{Jz4m&#f8xOSAe1S<Be|DWp)W8jiV-J_c$y$j(5bNwh$`Dor}W{ENjV|;h#@Z^}$7Oov~_P7nf*#m~@_`%*}2>}*B_+8%THq7E(#w`g7QO__z?^Fh^6qL?v244ca7m}D`Gv@t{S<=PtNj?ve%UCyYYNJ|oTtXefj+H|P>uS)~!x2Q+QLCkuEPy=eK+P}K9&&#1BJGGtnkq{C^D;`-XfHdi3?y7h<%Xf7s>Z;oKC@{?q<<A>lKz~w{k?e!SP@Bya%<L$lk`FT1^hq9xIye|ZhNasA(-}VX&2=2be~5y!(+Yn48WI#E7HTohS37jJO3<yy$4A$*<8<#T|2~XBEM1mw_f}FZ|{V4IXx?BI~hU~swuxe<5xlXJcDw1KZ=R~hTk^k=uX9yyMNl+V!6!8bRk~IQwQ!nC{+1ycq@ozDv=?&%F~=51Q&#egipf=x}w%<sx2@&f(8@I0**9PC9zxp1i|wfFUthWVi6c25K)sEqzf?<WI%y;=Ks)SI!ZI|w^^NdI&db`gq&<F@uvMA;GoV9v5tBG$uRmfaT9ceW@$`ah`YkHtrL;+V$?BKiqrceb82U{;EH$?*_ux>XQDf@JGh<xKf;LoFd~FsP!9^45|VjN&>v)2r!l%`Yu(9&u%jT(E8>Dc<;8G%(U3O1FOrGj>-qJ_&F$I%=@}ba5;QFnM#lT5P#Cl&n6E~2Sy1`dox%SPT@@WWjb55&^*(m@oh@ec5A)5r1`R+WV=Z{t9qp@WDT?{)$<iiy9n0JiJ!P6d%1iDMf#6)d_+kn)f0`!2iIKrL0C>}>S2B-S{f3UHv&dTgHdDE3Q&{y&nZ}x&L!<w-#KR7ie!8>ppXWB8Ya9KeoaTR=&-|z82RC$LCoQL38CUwizNKy+&HU7w6#V#lM&7cdgY27~@ek*B>SCT3fODRS?Dh>sO&sWHbm5@G!-Xzj6ZX<?bKdkDB*f(K+H!~g&@*P+oJ9yc)0`Q9zl_Od-&x2NM-I`c!v$X8bNRwMvcg)naF{(l&K`$GL#U<adt2JCNyA7u9*%5Bw<0y2$Qj|#lddXDJHk>ksP%SBkEvc)d@HCqb<LlS5cIrTTK~v4AAQ{y-(#neyb~jhts7&xz7ZVWZ6}US%iD#;fh6xnii-4cvL0DESV{X=j&5&`r=!?%=3VYx49nqmWjDQ9lJ*<w{YG7%TqTu*$MHD1d-mFV>+BC+*B{ca4~v)GTJLRdQEnVe+Wn8(&4GGwsf|~C`_26O<9K7TU#vw|Ke})A&$3#pZf}Xha#M`$FW!tE>(!I+!QzJ)-jFf}@s|1&94yyA`tObINPJXB(cSvduyB2F{m^~&Ne3(0-Nf5vt=Kx7KKd?JgWZF!kvD3I+WRzIc<aCIYPXqGe)^zxn!CQt&C}s~K}`<M-=k+6wOHHuyt|v0_AhRdnf%55Q}44^EZqCIH^cFdxBh;*SN=SGk6eiL?E3V0S-YAp2A_7%R?e3FXAebnaM>Q0^|ek;_P_Stj;q_3jmz-E$9nl}ZE#h)Tk9xI|NHSr@>1EHKE!2lYa^O_xO+cdeZ9U~+&x_`)O*LnT>9xX`t%Xsm*3WNcYSHI^|;YWO79y7el_v@`LMqG(B670U40g(gW&sSCAYTgQ`WM*a8=olPE#dc?BwzJS<;7<_uy&ry`LJMk3ZkbrQ+>4`uP5yYV5zX?o!44)5@oD{;+*yT(x6`yZ5Kn+~MlwYW%r-pgt<;t3>`L)7-d9ygliqm8Y{<<zX$EevD}=DLMXexF3mMCyTNYf2_rm-Gjz7c^+?w{i~<s{oQ5X_mnv;<a5_6&72gB?BB#ncir$==k~oAZxpxNN+q+Vy=HRKTd&#uTx*HPC&}vN<L$<EtuC#9NbieVy~WSFxNrFId3jO)jJ3|{8>3jJ7?nTIu9uhN#rksOFjsuuxEK~5x69|H*Y2x&64M{H-_q;hi(s*>oIKU_(lB>>`fBJ!wUmq0<ni*!t9s{OOUR4)lgj5HGD*w6ZC!~U?jGJ|-#=6N<@26lWLIP7qkOwB#b2s-&Bf?OYg2lc+PmeXrrkwToBh|=uCXgklwQ43X=aWuZlgD6`)RdPjkfkruVN$h=%g`?)cRqmor|8wQoAqRYh|N&^f1`%8RJZ|HarS<o_vw5tJ~|s-ShtO+3WlA^jT{cD`NaIdRGuzyJ6p2@Z|YUt+$ewcj`tx`%&I{SxpZfj{KSPPHJznao;a3r^rRQzn0p|gOy}4U2hb68=vv(>+E9fCaz`&4>zlKlY{!}aCa+nG<u94U0oeTbBWePf2-3Sria}@GID(<Js(a>jY;E7-017C&DF`nnbNx5+6!;L?eCRJqkjJBB=|5{^lu+smKS9sI&8~{cwzM@v$)$E%f{s7@bsp;ycaASsQwGJq$H<%5&8YLc5EDMo+!7$+o#=qy*{q>;<?~<a+nIYKi*&Ok8h2)<CEK#sAL`v9ydlx_+Z69IzP)C?8j24FRznHd~{#hs)}njPd6JE4`=7ek)a&yz8qffF29A-ck#8Yv#m&cax_-2)6Le#Fqv(<b#l>-;V`q?>og<2>g&nTVbs@LQ=VQ6$FJL)m(e$4<*4~txfyplqLjJml@7}9!T0<?Zx2uFPn}kIy|NhgJ#7Zp#w+Fe>*!{@`SzU7K5ZGvmzLbFUQRmgi{AR0zp%Krdp3;}OPTQVp1AU{IvD#N<>K)mzyI`_zlmk!wd$FA{no5>6FGhSoKD=ehEMH^F4d#M^?_X8c#f<kUf%bv{mtXG=g&^|=t(ZGT$WO)X?gwr<t+G^E{=-EX-`{je!gtnZ{NKtr|WBL;bSRvc+k8!Opmoa-*zv1(C&s)YmuYv&C18IviO!-t+j5&_07>m`RO%o48nUS(}&g7^x>0leNP)a=UPvx4Ov{g+DhG@ZbnuAL+jw^<!0k!cl>meF;1_f`n$0niI#n(Qv0qscsjW$wB<83k-shM`D(Y}@zH6=cuB7wA4|u<$D91_&0;g3@%4IZiR+`clTkG2yH!&0m)+d$ZEfS={_wUOIe1&VyV4t%)7P4RckFMsBTvPP*5gO8Bi}rIOd{9Y)j_+JKRJ0`-&>jV3(J%KOYtpHlPckd_4CtXU&Hs_=@!!ccC_}qD#g#5$<y7|B;LzzpDcbZtHt+v@I|XMyBB-vs?ph!O1roBQselpQ}azvq@$}^Z+miHeA|i)hqCct<SSx+GI@G0<r4=-Tfx-o#&PuVw6dJOPkyxWr`MIIho;>1uT^$aQO&4U(g#{Tc=x<=a9+RJSesUr%5`hBdi7X4*80j@_j=`e<#zijk{Xna^V`+hYA)HRH0}=MlhmW%*nLpfeb?1Qd9)U}Jh&c4T9?tI=uKkx#aFCod$Gf(6=U4U9M*iBwf4$s>hx?c+zGEgN8aN%)#3H^UFsv>8pn;dQaqEH%F5R9ee|OrD;&0R>D!}&GwtI1K>9346IY$}=;BjO5BpcQUowkp>3VK|a?@VUuMJi%V$bil?_s^{yAqYg>E7}3gA#qu^-kNnPy2=7Sd1@*3+?Io)opHh<>J%`u3e{EYn|szxmNjX%eCXe$J(cIsO{Y~+NXr_x8BsF%GG(WzWLf4^rQQ}j@s`h&h)dCvi>qX4ThI*j^uDyKR+x~q?Wq+l8c`1XO0HZj@r9w>?=~*xV>A+>C3O@o0rXfpYa%}rDC<$LZbDwTTs#)+l9+^{C)B1V|{ILfAw~-vY2={KP!BkMi-BbWNmXe-q_1WPvhfEp*&PpUl%JIIjw$qw0W<^hJLZ<>!zNb*N(Q<pVn7PYBAZp-<yj5aWEl9k4x>#o3+aG*5Zn<C?*Som0EG_<vF)3KaFyS&-v8k{UYC3Ijtp!{_1*u@P1n9$o1Q1Zq$3a+$fHds+LOaw~jSqYdhwz=F+}wWVIVUKB{dJDqmkrY}TawMM5cdR>p+~={R$dJ6J0{KEEgTvaQeN@zrWVj&G*?k@eG!oTNW>V}rG7q0;<VERK!T<Mp=xv;3ThSF4{7&BDujGur#Ce!Olcd%f8H#<Y{WibhtmnS+<TOld1v6vNBOi{fb1zTRl<MRvEJ_2N@Je!AGZNwpi7la<Nd+D%mZXf}hN%dzn8Bq>L=jp#+Pu^oB7pRU&8Vs5#%wiSL(@2%hLZL9s3`aX?CPgCu!WV-h{{t%yxtx_bLTTJb?)P{UeeBD|}ZJma5sqJzmxanUTC3cf*cg>_yTuDV$RrBS#uUl`ecPZ09u3U&wEiQLfeGg|jpRScf@ojO`B1FHjJUG9sUk;y#FTGSg);l|_@7?4tj!qu(kMYW8Z`!zumM<Rm&ntI(+wb{drMIbU4+qmhxOwuF%(pJm_4M6x_DYe$Qc>+>{gEv(H%h(?qMwK7`EK&?<S~D%bq+R~&1ojGeY>~j@2;<94voW`HNBC`edy0G%AqfKXS}PgkH?8fT3t_9p8eUY$wsNXJWe%z?@D|(oL&4}8^qtYYvSnaV0l_x_O}Phd$Bus8o!<-Pu9=(U-#v<SR9m8xziP&_SRZ@Y8s72Z!dOF5>0VeJ$k5C8jE`E;dAAr)feB@=k=iP<+A?#__X>l-0U92?~Y<yL!)_DIIQO@hZnLo)j!jvm+{H--torUNBqH;ZGJxZPB!kkd!M7uU1F=gwU*F=(rY%oUwhD3<a+Y-Tx!O0r&ont@&0=EX*c^KKlCe6T|PSL9=<$A7T@pA7Vo=_Zuj!;Iy1S}_S8b+v_0K8dER~aylORXj<tO$pB_uT>-~5tbGN?vkz6UR=l2ToaQP}<-j{>b^~FWy`g23i-HX+O#c4O`+pm16>m&c?$I5AX>nL%%y8V8V82gu>p02}2ym+Q2Pa37ISArXJm;UgmupHgGxlKoVHTCiHDY98wOKd*m(q{+V+n341Mt3i`-g(HJt@nJ7#W!Ob)aA)ZYJ2;0usvN~@r}X;VH5ZAXXHY}jz&t(H|RX9W=8pg$f<t(m>3cy_jV{9baEN#p_ECTel}Ct^;W4hAe5vRZ(n~_9^W72w`}A?-_<LGj*YUxM9(jO%D0<7U%z=1*|_SZ>U!%Xvb@`h#|m=iWb62DHGZsLM0{!AQKCKA&%6fZiE_GHzj!qc4_o0zd3#+`WWWBX_b-qA-H*!N^Tp?0|FR=Kbsja<*Ammc<!ClMxy*=fdk@{C^KSCU4{=QDlak)u+*`R!8~**`@ie)3Qf=n)Z=Lt-e(Wk;NUusm!l{(wlZTaY{@z%VZg0-B((du<?x&i%JjtE}@86CN+k3`Fag@2aOIQ7O?^WX{MX=JRQfXK3?r;1j)8OaJ{^INFq+Aj!+3mAnZBpG^&A)7v!_VW>_`~&EF<okp4=Rgmsli$2X8Y}|aS~nJxR=+;<!N-}yLmboWGCs@%+qr^cvbJ*`0kI-dh3VdgWK(+$j9}E{`kB+sl`9_ps|&!N7o)7dcn~}Bl3K-xz~R>del~~OS{SH?(Lr5e4cho+U1Rs4Tq1Pj;bH)*=#baonOTACkZikG3rm(vyD;l{HWtsHzJXjjl^DJ(plf#d)rtKPJ+r+<G4*Y#r4%hJ^5Iu$5I=YZ6k3Q7pr@3qPBJT7A-D*9HsW+H}B)vMer&dZ3Q=*gcE6AtdCOpwa((j=t|w%J3Ts2jO)F_<@jFgY^#~seSO*QPs11c;rl{Gt={$quY1Pvm00VY9mMsCJXRLp$Gx<gyo(iYijRFYwl2jlOFiFY(!9Jr%ARNXohxn7IZCxl_xasoWEhTBjA-k*nozGslgI4&`N8m5x&1h8Y#Pz4p)N(l>S<lkPoC~CYft{NdK27>l#2Oww|=MGrU<LP(tE9|sb(uyId1fe&*R(f=xys_@-R@IONmd(|FCuvlOFv?h4y>2aaMS3m(#gvc5UzdU`;CAyuQAxCwt?<;dAdMGD#@u{&~rl*zm7?KHW!y(fV%UJuJRGrV`EV!;k%^R&RTMy)qeXRr6u}rWforD;Lt$b;c)3`N#LQi{NN0(~@t*)uWueb*dB_>rdl)e{{UHt8LegeWk-na80e<ZRV~Xf>)PE2hX`-|Kqv-aX8&PUCd-YeIJLz*w*4*;VOS5XOw;Id^!%^CTa(_YNBy^B6jyO6<_s6FO^q6vZ<51@!CV8+7?^li+cJjms7=u*L-I6<<@w=%x>uM;_CbPW>vhOzG&KZ-zb#Si|gq1Q7}Gz?Wa!fd^d+1%K6soV)^{O5g*=OE|!(xxj23xcdc7pc~1?sr=B#{wh9O9!$!AkNYQa*ud}+h)j64@YLR#UTklkSOw=C-kIBX8$@Vn<bWk}emR^?Q?;o2*<@WSa(OwTqYs<;YY+-A=xZDpO*0yef(a+P@gT0fN*KYVGEUuRZldc@xNZyv@_-ONC#a9tui^;QMw!b}TeH5z~lj6tt^{#f}yE{pDjZJy*e4I(<E-s%(+0%Mvx&LtOJ8wlV^zFUUYwY~$v%R(XSigT>JI~jR(q-a$e<S*NJBgmHUB8|OFJ9`|V$5HREFKS~!qcm9;XAFQ#)J20@$t5`QN2F$`47gYiE>j(Y@bxd!SeJjJMJ8ZtFe7q%&f(t1>a3G(tb?p>&djL6|~ImUFI~eHTu5c+WToQxOscA-#I^A@4mm>T%Ywf(<gel_mp3`TUB-oi(8$fwtd_=zN;VlP71kxR2}ch)9J_4{X_pD+?I;DWJH-<oIRzAXQNkNe}8>*Gn|h;?D<}^YvSS5KfaHNWZElQLCLT5E-#+f*1PBV?eq5G-cdW{+Z{yqqRaX7)nIS17t7^uKDMtL_0+*#p;0;2-mX)76R~?C>8o!>Gk9L^carIkMeQVeaaoD$xs%FqvUc+jsimjeEBDLyCoyCDC9a<|?lO5*&MHA;^l_9R6d~?6vW1)Q^XmG>q&SwMZx0Vzqqy4My}VK%58o!~>~rlTb*@~$##dy+sI|hz<>g@aRhEvH^YUr99a|^Tr$4*cUi=(x9d0DjcSo)L+t2*sa`-|`Etm7ppS2VruqT_Rdn?hyWPZ7^AFk)Ns+9{peAc>pSt~pp-j-_R=lspddMfey`dNy6_MRg~G#OsK-OC)6GH;o<`ssT+T|FpQ^67yX7nAMj-Q9C<^L9(#_Z?h3EY}_{o@;|g|FWK~7`6TE#;Y7HTrVH5G-9W_snqe^dVRNd6@S?i$BX^_#q3)x6<*10l{Q*igXH#GZT&hvsZ|rDgTwuF`0;!>c~-96JQ~A?%;sL=y*$acwm;t!#gcMxaB}fr==)DkiPW*tDyXH*<mDi!4+*(>Xc)V1Z+qH)a`YBm-D_MCnQ8ZZyQdVdU+$l--?F{%WSG4zEq=rY>#K#`v%AH@!Qy+dd7|wVSNHaN(p5RNcsh9XR|azG?D+WOV*GlOYc)#6YED}lMH5E9@glw-AEq`dm0&>~b~cU0yLdABxv!2};aV`Bzce-)xrf2Dlg(Yr)p1Iju5~xAv*mc{cz^9cdMS)PkMzRjS#)2#dz9M2H?=c}z8(1=t{yLrq~!W%VRx^7*H~}I?{-_ykCDt;eC47N-rH*Dwj#TChvk*C%JZt$i{!S>_s=&qBFt+4AJt~Iqb#>Y(PtrmL5&HvU__<0Lq(AOLJ$QNq!9%an`b}u{deaL%DvZ`bBwSnn+a&ce6;gczw9rekS6x|h#r}X=MkDumJ#dCOm$>QZDV4ar4Fz_Jo)PI@jN>fy6K+WsXRhvPW?A}wbC;}Y*&pj58$#nN;@~^NFPW0{_TuJ@if=acx?hor}O)BnOEI-SMRjcs*f5Vzb7DIsA&_WEj5JOyNk~*2OWLlpU7w``-j?_wLYC;S2JqO#-W~OFAv21O8KlV_>MU_jn8n3^{lo6a87Op<4xnaH>yxz&1c|k6XeS4wO6C{j>RhYU2ul|-fd-`JF|<UH~xxi{lEjMiOv>{zP58Iq}6!(==x3FXZtHu?yKH!I&8eBAD^e;+u!;&0A0>QwRtZ1U*%k|ntm&G9kM2Wdmp|vxQ#bU!o-ZG)b6{JyR0|bHE803TiNPW>TtJBur_YBE6vp=_*N$%s(UZ%$DBosLg0%gwcDLFa-J!sF0Tse31BW>QHwLu-&~TOaq6Or&A#)Z#XejiKF8hyo?H*scb%$m$bMdKI9l(j!9)M99?TkgC+%FPwEfSrynZ1L^4V^63E<7USCrZfUJ4u!IY~4sNk2j$%VDlO)w&1$s7qsKE-v#Ht%Vm-AVp4GRxQ|Pkv1Xg7ezrk{rvrq`VEi#VeB-xdO?C8Jj%i9Z-e_=&_*|0)yO%Ny@7356y-3v1DjJkIk_60|M<_)CyDER(2EiHFLw4KQBPo4>OR@U<=JOQIyZ9RaU2-+Ge6zsUJL1=)M3~Ta}4u1R_*rkxe-LrAstt@m%-^$JHS`m2;Wv3yxi(c(+>=J`SjAQ$SuhXQ}m<$NT<D@L;)oayX+1QUeCeP{KJkfMR#3`aN)MKLMe^z`R?<BGG=4Kjf-{d%Ab1YYU@*b_cTSQJy3$v4AXL&hi)0oRoD~2Mg2rJ&sD#_85bR(H_f1R;q;??KZ&c!@j2S|&0xe-Den5TtJBU>k<iK)U>`egZo<8r?xE(<5Dm=&d8dyoptbNmzK^ffL|c$V=VA6{;28J)MJCbROQ-U9WbMk0=JpTvjhw9NUeq*@qHGBY8-r|VK<a#<kaUxXp?;K&zl(ak0UWBxgiULY)>gfWuQT&OC!d=RU+495#o+bh`}3*WaM%_n<1cq4)?&k6cK5e_;wX`VVP)OBZO4C<tD=K-IxNG-l5EopSdr#g>{gPG-nhsYw3?I8cFb0nxdn~dt<tJVbR_*=;yGQFhbgf3#r5U0lE24seoE159~zb%vWCy$BUF2<YTRMY&Tt8e(IuBY{%BX#`F+g>0FD|p;pI8}qAE2j@1rrRi^NS*xwW-R{uHO!TTRyPuab5Iudo9Ii>*`qz4n6aaD94LEpsh3RCQ%w-RWZOK~2Xe>+z+*H0J>FsSmbyuhk6`NIMVQ?1KJyk8#-BX}p^D>L=q|GY|b+d1ZO#w|tKG&tbTPJIJ{r5?c%bTfa9M!1|==jjY!0yq^Z?)WJ;MIaO`B1^b=V>#ZZA+TRqw^w>BhWSN<-#dRl-o<#8stk<-v0&DQd_@m2AA%#P;)tz!?F%;&bR$L!i_jBr5$l9b<%+#BXSAzz9zBb=w!L^d^uS2osj|iAy?Yd+q>1PUdC8&1aWhJ;_F0#*tjwtU3?AzwtC%jwY)Nr@ss*fS)w{hgd!JAI40VYyAkei#KJ7LwTND)^y?BBZmP7~Xr{nefN6!m%Q5_*G8oKz~7T_#t6la4swJ8yy~_1#+aO@?uIaEo&|C?9tHfrA)bb6x4_6|L~c_1dds<L|J)I}jf`3(c23t6xUKO*FPmMBLp*)Q$6dckQrOrE*@aUE1pDO8ahCaU&-&STx!M3U#;f@T;~Z5V%5vXT|>wGxW5Fv70GgLaLQN`YA+1UG~*s1AAV3ia)_Q4`#3P@<F@Z+^-Js=un8);9z8OzO<ryS5k!?WhVDWW09&(eGEe+_=xXc7lqZ8Z=@x)oO~u{=D{!V&5ELxOF2c^qgNNKytPcdQ}c3bp<U|Tz})Z8sWO(A0huu#*9xx%2tvMqc;IdI#HHLJiudgBqY3&CzOFikS;^t-g*wCq$RP&y%Qk%tgJql!s<TK}4mv=8lhLV4(dkNCHV>>q6^yb+k#>>_@m(vMk=O0*U~T>S7|in>F2PyaTvu}bOrxPGF{e-VJ5S}N4PxE>O1d5GQPCUjD{JLuh#0~vE<xO=;%r-^5#Ta>=RO@W41N?Am$Eo6zE3_#^ZNt6RtIQP0jE^8nmxS%dZQa2>yqVY0S}`U`IwysP3C|>bHC|u&M^+$xw1Qijb}Pc&ri#Kwx*Soyo>avZ;&VE$(UKV-yTNe*52GtjcDW;9W_$mvc}nC<US;rqI%zh=H+`ln-dAU^JnW7O`cl5F!gNqja9VEG&lfAZua!Bd-tGW^(+hF3;$j0&4s}Ll-M+-MX#!0Z#b}{j0U3$mi&Z^JMu33X&go8)_F~+<nsKkSGN_AU!G!Dh0cw<zXg_Kef5{p#S5#bYLLI+Evt?jcl4EGcw&ep3EB&y>*XGI6-EO4ULLBrn~lXAGC1!me@5o#jg8wl5^|B5@_o<VfJ*y2T?@5#SIdP{ui=`)oqFM$a>wA4sWza+Kaa}Pc5k<#o4rkSG~8U9J^yd0!iInYSvWFo7SL-RD%VH5NE@3kbYOSPXZ6#v$Xn}Qt0D~A;%!}{_?xzbl|w5;pcd5&-mA-DvH9*ql?-~?;3BMdYH>r5x4LTEcN$<{mU@a8w-v!uHcM?6ZJ1GevF|K~TXy*76t(<{k$etsoFR1X!}9Y7YkPF-ZZBQuZ)U1;C6~6eun;#fdR*<4L_8W@?ll(sWHV!wc?CZlu7!NmNqD9K$-L~AF}gsz{M4Jc&2TrH5V?6aviW}1R~NU|yV`R`7fn7iz&^7B!1CxTk7zaGwFk1b4gsQ6c_PI+q&h`<TX+y3)OPYS7FS=KytMKcOerSm?w<8OD^Fi~pIvnNdeT6?#iMy2eQ%Y01*oFJ$!E{Itg-iE)t%Cc+y&CkpEA_%@)K+t)Sp^2GfvpD8z&0~X{Tx0-}mGe=@n=st_~=2w(P@C&1U4fz0c-E20XKAyLDi3?W?W3$2C1IPJ&03Md0<Pe?cp~G)e6b?<~Dq>XFiBt;Pkc39YmIYp=@{e%__Tu*LmOMd!2MpEg19+20~=mfgn9%Nm|9x(FU_{%C@gpshDp*p1bCzl5Y4#nYzo1@|)78<qw3g}dv~aK*W<2H<u0QU?IP+B^j?S6nr<o8@rPzh0|^!$Vrrp5<(?Fws+j5G@FWxW;}@>h&V7)WDL8*VAeJ15vAD_JemZVn1%&3rYMLaw+cq&|Czyf^1A4Y`!W-o87x>Iun=OU`4w4rrVanqz8v<N)t()U;Ic4l!|up>NURKo*Da-pFFEwX=ZJe_eKRT=}wxGsO}=KR+F9slN75Aa|J6fTkGojAhr7{vsgY&pjQis*eLkcYjI)ukzh9B6?kykzr7|d7tGSSoP)!)jl1Kr0u668d<PN!_3F|MHnB3p`mKk`Xsp<F$u%34HrSiT7g}3v^uEL!mh#<bM8A<5;@_2jaIddc+{CJ!w$s0v9sJPs^u1brN9Ru<X5a*mnmu*&S}jH<;xk`+)tOUn9t*(YWtm_G#h`U5h>bTD`*md{e$q?tPuoxP@%<ubqqK6e+LsX*B;PHFXYTpl05O85aN(p{!qL<e-|DL!;DOB;xknW>a-y{q?J%A;Dgl$XQNKR=!*X|DoxB8nlK%NRBPsNI%3~79Ek2L+xGr=8?b|x0XmFHQ)tS04NfwqS3{oDBlZ;}&We)_a?1<{m*00H6;&yhIq>+se)@@Gc=yr!siMKF(VcOlFwJx9Akw)9%wLRCfcT>h(Nxg<+X?dQmHd$-K_2oEGCA?qxB8#W}YA4Uva7spmv>ttTy}fMt16*r+w-?@CT5wP)uhrMqs=3zcmrOp}&_{wx_yO)5FC`zRz1Mbrb{5|+Aw`MmO<$}C$3V5Ze^})CyjKkx&`F(+j8~hF>WW4D#-I<hn*8JQ(W}KfCgJfS3CEl9(K=k4n{gBPCcW=tC?2P~5x6g$A2DpUz=|S2XABdpBKEe~6OAq%^&$7re2o~1`S!lt)>vVUH5Yb3oN0}+!>dn@Ca$uxOrbKqX2z48z~(eh=yj3x@$tlhHirtf)WVMa>D}FE!T;*yp?-#@Qg3a}i&W48{q}^M@wne;JY;sL@m2-xgReStdU}pFfEl|h9je@8Gp)^9rdnGb>*Clt>iW+91U@(^d{+BaJ@H{d$?tE+!nw<SJHfVb_c;8@za-823J{=&&9icf_o|kFe<A1eHp@WW<ZX4{v9jyiFFta68p&<QLGjY74={xt>~5jd0)y5FUg&6FqTliT)Kj#}U5SkGw`m?aCur7Ew@yAirg=MC9JY8&yAM@7YP%sZKFF<ia{1lf!ij{EwEH~_tY3c~qU%ceurCo1jJL(0jk$jWZGNViF%asr`F39kS8<e6@*kdA=bFY(A3uILVk=*!FPH*r_OodaFu!OW(^NaV7n(CVUcn`8{1x#upN^tdO9mU4LF+P}VpE+!8HD^(RI&V?L2o#|WqHLo(XqeWOdY}R_Ozge8p;tL-OSmp2~vB4U9XibYAXX63Bc%?_Z{Ja1^jFOn7ITsY6pBdrmb|MudtoXzk6S~_vzLur`znk1^-Gq{MJ}0t`a_?Q%8zuTUqb$!@0$=*&hpkx)M%=_#4+yVKu1NP!->a{9}|9JLwMmI=h%Rods}Sok*+OIo*c~oje;wfg<RgJ^)C!JJ?KC2llk;uQy=rMc4fKo=x?$0YA(7-NXPuk41moSB|^q$k%nzE$YjuI0Kd$yS%?8ZhBfZF$Mf~HeJ^kNXiHgM70<UrmCQLz}6(F`?GCEo4p-y_54W+2vU`uh5;L!a>dpyxPLC}*f6W@<EC*sK9?@kvCB^?o9oqk2p?rGu7AX~0Tw-|RA1FW6Y$hb2)cyQX`Z(HbRQ}$gq7~QPAfZ1kTlXcX#T@q2hZlNxIJMuV5ot1uwRd^2fb3<UV6PR`!Im2otZ#UH&?D!57^^qFHA0rk%Qnw?N)a}u7D`qS%;sCGEjT9CpS|n&BNLLxdMc~&x_wM{CR)=_Bv8m64a_nSGg<MfwSqQ`lg*7r>m=e<2C3&e?5|@(-{kfy~pGA-3<=8{WM@-t$pGTF?n|s`*j05GlX_do!K%J(u#<7I4qjIvEWpcuRFl??I6h|zv|#YQq7URngoM!=Xy)+x<7jyuwdCX@CnuZelO`kA-6{m<hP4gcVjb;*4S&Tpz5Blf9L_TL?nHV50~Bd5eL}Hyz!J;3FYy7BlqaOuO6Bdt4-rQDF7Y7f8jbjxs<&qs$X_{GrQdC^G@06lO)~dl!x+u?QV%EJ2JDBM5LQLw^8-mIy;R@+4rxb{RLV6l-;ti(I5(ZhW0rY50+1S*W#Iqw|_sNe3(N>bGJ{ukAvVXKi=MEy=WX}_tnX4e;eSwbuJqHr#ddz*kYCati3uE@h-WRw&eJ4F<&TB3Xg?po8WMm@gg(at2m{y&B0-9NbU!>hN3l|nPEIUq~su3r1MA1nEIkU`YyN)5l7T^c@&UMHZ$IQ5NFJsVp{k)aapKRw()#}Qc~?v>z<!81<}OKHcTgv{L-4I18@E$c5E<xkjha(vtw;AV%n!S(kSp%@5FW<<?n&w%kihY=K1enJJ>dS3sk(j05^t|#q)thPF|9R%y@iEdYZHxnZ)z=Mx;h-rDLK?5#xollOaCD0c|acDj3m=ptBK#TP#f@eHVOdZz^p~_QiR=c{+n^f$4%*%wBF%=Im=F0aFzR+X}of7QQ?~aI7|tPeJ3IPd@LsC9YyOFu#rJtG`|8@!>vO2p?*XE<GFQqJr8|J#hAJo42v8wg7PaQ*xLq_y7L+ui5M2y?zt_?;UQh-Ea2W-T&U<Hvj(}E+Xody?-wOaX0Zs19l&tc?0#9=z(<v4Ii!mKt1jIld`f1i?{P2no&aq#KuC%mEZ*pQe9Rz)%!XfsM21d=Gy8}`McGkJe9BhJ1d_9!Py+&3&@H8S_DiO3@u*^uKQWHv0v%eQYZ5^_>`goDoKZHH?+}k`}Jr}slDIbDU+~3?PAHYLMM0>7DtuO9D>AWe|DGvvFm;7g!6vCl{d1kW{08mxmbJb*u)B;rk?#db~^75@1G0Nm(Dv{^%148t@e}Votrycr~GK<4I`ua@xOyuPKY<%0G5vlPxhvm!2pr>iHGmaNgZvDLHn!ubNKt$IK9t;Y@-6zj+`wqBAaCy%%ba11KIb2I(D67E&dE&5linH=4_H$b!gdp-#GSbLR0eC<0jZ;ed{HF5^6k&XZjl)SHDT=!UFbs|8?K!FV33Xq&rdJN=0~o+zSsy9hWWYcivm>a$nlP?O*vA9B0i?9kDNH2JLiVnlpcAVBgzqkm8B~DhK_9d6YmiC#G9t?|SKk^+!W=^C0GW!y*-;KJLDd{l&sb`@JXSS6)Nxqxqc`q7ok~<shmG^Ur$WDcvI})#I8{i#W^Qod~ki%*`U}NYz1@wFl$$SzYv&bVT;lTgL0QCnpPp)$Q;RJG=VbVbo7OeCL0ciV&0^q3$dSUq-!_QH_s7x8XzV{N-z-WhtpQ4afk4Ur`)4s|QU26B%FxYFAm5?WopsXAb+EKS`^a*Fgr@#Kq7>47ZOfOfXGS)}z_xWpD!;6-sC|K9$jF$hc*vwnD$reN<7)F=cNb$`{B!bFIkeX^>pG_V~3exJn`^PldpqW%q6b$92DwESZ9g$XM}I59)-Kqu;-AKl&8_t@-4(gXY795$te`V)JE0A;^3j#~0E<KP<>w-QgI!|G71i=<ZniN_nTJA|LD2puqR<F(k);STm(=rR_4~>b7=?RvqC|Nn_>poPCYwo&R8ZJ1-c1sbr+(rhn)mQ;B$vhxVLNwim{*5d%m!@DC#vUhCta&!Rn`n-1((fXkb>4Q}Rn{w^0}AJ~4X^U74<CMH_#t!r=1o!RfhW`M4*O}QBrvOkX8C~U1)+2)~dlzcKS<-tR~P8xso4`3s6)$RQ4dF4H=$`@{oWtbj2KB$IY{>EKy<n!I#K#KOgZ({P7d`^GWwXtJxISKjqIq&Xj`y{@PKp_EO_+0z<?V`KBcb1XcDl5}P3Rd^;IkEBIc1(b`o(w9z6LwqbU2Lb1-m5#8VA|gqiJipS<qPgfaZsqE<xl65rv_+G$o&fsu`{ubi}XWm5zVXv-O-6Teb$@nX4`7l5zHS9OW9;QR-{l>tE$=a=|?Fl)Tu%YR~f`T$Rsip<y!}E^cYlTQa>{(s#<Z=54_u9PjdEjr^;^HqYa1f!@ILJla)BANO4B9W)A^ePavrV{iVLQ1;^Qp^sHYEdbh7itHz3>|4jYGPu@BhD>OSa85rvylwCyh%tWZ&sz0ucwuOkvPKMi#g?{Sq1>@_MhbQ&>_c#zphh}w^-l#6xX}Q9bCI1qv!M4vkbQG1x3o_rgcz1ax*I|;iD!dx()s5lKpMzF{(Ki@{0)gf>&(I5_G2pQmmku9*_8EVf_%ZdI<E%k|qP+<;5Meq|!N2aGaQ&RxJU-vp<AOn=8+iFuPPAc{gU@!~L5_y{##!UHmis2?Ly3jeC(pH>)~pPNw$9KGPeZE$cZ$aSFGTc>k3b#f-BRNTWIsR~#+i^(o9l}Y4G!@$7*|lnzo7YXc6zDy)XaeWDJYL|!@AlN&khFtZ>96;TL{`|(oGy{wUv&%x%a@Q-+4p>ztVJWyT_!OSg&HS&P)M2nU~hUZ_Ngpn2io&UJvz}%}L?$TIEJ~gohe_Z<Fsy`@3?aNhpWjcwOULKiE4*xDM6TA!h6H>~=E;5_a3S(0g1Cz2w@|$Fe)Zr#-+V!Y}K_9SfpZTJMiEj$ZZ$>Bg}>(TYG33$8S+mUZEzy3y>>)xsk8hQBLQS+K?wdJMf!C*=JaiC+Bo=b@f9z`0?WYqVnqHQ;FUnkH^RdGRtR;XH&Ck*&Tu2Xl_D2TA7wLeN@tJ=rYiKDc*(qbpJD^LF<y^B$LyxwVv&abC>D@caWavO3GyqmfUc;H1T8^aokhx2{#V+LC4#LqLim`lWw35b4&cj9|_<gI3$>Wd10#lN7GBi+x=0Cf&z&#0InRg@$05Xzn}kKGyG-tGv=Y&pxFg+GLpP)Rt?aetaD($AdYYbHbQVkS&Pv)Q!xyXKy<%WO;L(j?(hMB0vNm<UUzs)MUKuU|iZbhq7IkH1?QF9fcp6!%^Aeu=nXrilCYHb_37Nf=e`S7}rVBh4;(xk8HhJ=uFpECrW<~3=P-L5aK7Y{yx&L9X4ykXJx5aYAEXbbv(3YF^P~hSs>%Z0bbq(Fn*vU`sgj3y*&JNYxxek@bg7RPHUv{z+}3S-SL;-5vbF$C3T-ERi)WOSe*N*x9fhNyhJ*+`uQ(GEePx)t#DF*Hg8ZctJTm-zqUFPKqK&&)?lW7EB*TVcluT33O*cNkimr`ep<IfgN=}Q59?nfar;?qe*c3ZHt92y_WUCer=6Z4q$}49y>2%$6hI#D6_rW8!07a<5LM<vKLvB%JE65|y^))pg|i_VwR30ETJvUYrf$K$ow|=|eOMV1UN+f$X6T^Wk#qfzW1p*2f?w%AJ2A_Ru$!x7bGU`C_2?+m2nQy8&y?Cj{&axpL1*A@q9?F(ex5wQ&lh&9_}mcZjB=t#Rmf!e^HZG*qC=>6se-K*4apdq1i39pBf*y5b#B%_Mag_cr{9l<yf_e>KS{T#!~5Dr>Q7_n4Qo@76RWFQ)YOjRUq~Ph)VnajXZC)Al|I$l&xOLxu9dtscb%-QIj<dSLiP8npZ(-~%g^@$MltV(V3s1b3c>U0mCiJ6(qj^l80f8<s|=Cls+UyI!~XD^;AgMjX5MBy`xw-Ew3*MX`mOBw(d9a&USxauz*TlJLkojyc1!4yYOT*9^!9G@?ML0ytLyNUX~QOl`?xgvbqUV>R~Nj}mx0^Tkw0^`+~P8+T?U5`>{02g+FdoIhfj}NWTRF+PuB7Mi?2aoalc3*sMbk}e<9g9d{6yGH+g$iYsS#Ii$*QR*>w$cIqcb6pyzPR84>8sZM@&k(ea>+bUxikmO^692R|E69X0*XRC~0f5a5f=7#EhmU;p1%6+lwfPTSxN@P{>@)e}q-%z;hg8s0nD!x5?Oc}h(u%YF(kNyq-H*HUOlx5bRqes9x!L&$2{9}Y**CR)W_c5e}r!{z5Zd2qZKR_3xXSw)v^gZU$XA>8l*1^=Fa++Y)W4Un@*a65S5<#3ieoAoqZw}r=<7NFIEaTR!3?5nI=I^a5)OxN9rAQ_Zd<yOZYG}A6^Rzj%q$jwlCK_Xe&ja2yb7l4E;+e$3Ayd}LuXKCFQ#!hcJ$G8TdRml0}ixDmtrKV^H#V`9dA17)DJ_vUN;uWnK=ew3;IVb$bpQ^96JUiv1-6I@)O+puMcz7Zxs_U<QS4fQfHav%kufOE6{IpnsY1Q+RvObj(%)aRMY!M#BHIsL(aA|hTb;%7_wcSi!;^2;6-?)228$cdy6|~y!Uruk`X`Qe{34s&4lx6XiohRk*zFSwU5vxOkcK>-qKj-rz?21(P_zO0cLX73KeGncq$=~FN=m(|>Kx)@I-E$7U;gFlnsHl(5$iadeHda_0oAK@Xb-&K4`SQ^`yY)j3>0MLq^*nEwotm%xR3#o2Bw5JSQ!=;_kJ|BtiA?J_f2>UHk2=^@L%;$g@viEv#@X*L63&^V00ivpi+@9fa?EkdvOaHCjLQ2f+YAs})A1z+dIJ+H&f{f2$J=fMuetDYqT$MKT8hUh)reWM=^rgS0g7(>Y<2eeICX=~Jxw1{uUb#e<SDAIX7TNCLWUvT{+rzlLBA#XTC8I6J6<fB+-lai{l;Yg``G<u(0TD5RH{J+jn8BXU22l%0{ty=9lo;p^ik$79;=Tw;=5ncLgD{_d>X9=Y~hndZ1XD@9aBNrc+@=PRdef~n|}rd-D18B!BSmLhDY#nE1C~*I9&z}ndd4+K-F(cWfR<4<w$`iY(JRJspl^+%5r>L$18O<N<FSuN9Q|i+8gxl#31p<rM@qDZ~!AG`n~;#BqcrWzd6<q>bu6eM`j7n9mrMNTU#PLdv5vj{+Kn&rF=V08>_6_B6rSJzM-NNtz2=9yH@`_vzs9!J~|TM-(f~H&Sht>rIpsDV?e<*N>F}xf6gue3N3H#p46Ik$xjO=h(lAWJ^IA)eJyJ(tt;I+L^Y--OlVw(bKK$$VK0rkyC<TJ(uHdP240S}+qJz{4<Cd62(7_fKE_n#PWJJI*qE^>u}w67F#NFuX!luFf4V{PB*3K<0pPBwcN`B?8)9dF?M$B0k$KCLG#(w+b%7{_WSne@O$G1N)JpFxXTM;qT^{0owfWWyDHzm&8RT>0*Yr!=*X{X}>5Q4Zj}}TNR-T`|gWIc5&G}4o6MY_U>jSsjj>pk>4>kWHlOsmF%&3E}b|2$ZbpCvfqxpnA_JNHRWsht3F<4%_hOSdSq}09k7OybKY`&<l5auoJy2&zdMYOo%<Td4+!wJ7kTKDYL5|4&+zu<P%PV+zV&<$*RyU&NQG+(k>(Rkl4p)=@Reu*@tE;R!GN$t0=hCFl?rkA5#Ae_;{r%-h>s15dXhTkc@`t3*|9<NViHEn*ieWn&K<%?4IHEfm!Z=bx^nXHzR6zd>shLU->VxN1{VO}!xXSHR$2L-=ZR@>LHH>_8I^}4_+(zgf3V7G5(kz~M=<NCg$`-d1<Gou|}4gVA>pC|FpdMn$n?mn|;^P#G{K7fojm9%d4d+V;%F86p!nX;<cGS+8w42l=SYin}!<Wqn7`dpq)mVw~fI@WvBAbh0!i@p?Z0-Kbp%TKJBb_?Q18hr0^7O^TUCaHUjo!jT-jV%f*y^BhXAo3Pk@u9Nkby)VT>?8y8qI38h3-XufjwYRi!=*ak9Xz9fZSJ@A*nNclk)y{&Db}Bx{_5IntG#;M9*32tdYI>g8QR$GmK<M_%73Mfh$TtYxfeBjKm{8~bWWXz$9?9#xr6y6>NX<QTgR1A`Z|DnL13SK<uMy^qc0^kZV9VRN&oi(R=4rny>PR249K?*vY&6AzNYf`-v-PpveZgzQ;6OTGRxT$-(ow%f}i{O+k@d{>ijML_PO2taoXcVVDw>ds(o?;K{Tff!P>U79e<cW9ktdA^F5<}C(jfwHRBzTWp5a1FE0dF%Ss13#XSn#NO=TbR+9*XZU!kff{}1;9@)l-Y=$}J(#s7B)KF8eXG;X83wt|xkEbi87v_W4qc^eK)kWv&Ii%Z%nSELW_~P{}dtU*+TAhZURCw&<#nqjx3HK=wh(1_PG+e)RitTcEF#(plUYiVQ9@d7#9=AA;5RD=B8THV-4K7SxUnE`y+NKjy)`g<LDR+9>@Uc)1YbPWb1_IZ9aR=(M*No#HTiwI7wpFU<r**A2hLv`3lsAF%#nc@ru+pSWyE3Jm+T-+C(g4zXFpJ5oYT$!Fx=BX>SYW_JjnB;e{<(^Z5n&hwKTeSWH!ruvc^6mQ$89(2qs+E<Mpq~(I$H&yV76%wQvHTc2GlhzHGq30_Ok_?-k?~y``gp^#Q-Ne>4*pCX6Lux;EMI!wS*j5Cq&N09h^3FBO{V!dzhY+G1YACvFrx4^Z|PRrGBNg@W}fvQ=WZyQkK8g5$N26>dLO~uXRbb-bK-F8@22G&F~M8AuHY2`RKW@DE6nV16V_UmF~^PLap=6XXrStcZqth`D^9H&geSTxZD8h99NyA3OE;h<DI8h4{jDuh)AZz($B0Z&FjT;_NBrV`NcQabUDtyN4{k|)D#SU(|Y}FdII8)^1q8@{X3)d-0ny_OCpQkaufm2AzC%0UgKn?iqkm1!_D?wVy)dJ#I)6<J1ciQLbKKB*XYus@u6Ty8Qj0)XLo&#*6GI3?@4rycRwZ4Qg%AVmo8h6eHb%?b^+N(9SjI=ES(x<v-UTiZ?g5Po72aKdeg^)^XA8Qzw*J?0RyZBK$5jaHQvD=z21Y8cHy1H4?mbW*?Gags9S?;%A)20M|3cc+6uQYC$sA;X-C&C{!ocY$AYm3WqbxLd9($!t`H>0<Mrf$qq*1u#_^B726d#aEsDLcnDO$eUfoXOIws82e2^|xu;Tm;(XYasm**qN{~p)$<RD@BG#!*ZIA`{9p{w`a;plsTFS*5@I9ONL>M7)%!r7<Vh_>9B8qH6MA>7@quTodrrc>ywMNK;};%AS{=O_4nkMGm|=q1=HIr=Rbu<=B3Xx9AGJY_hzN19RT`7W71XyTYWVYw%7Q+b}2&*eRN6XPi_&Q@Y>ez->c@=CT94@2&qk!n;nk8ZZjKJA-u1nFT>M|{QC_|b1-lk<i--f-Var4KLB*K%}fjUWe%pe?i+9XHJi00^&*m``Qd3%V!qih?!vP-`Un3xGCi-1D%$n~?>Q@2v`T8bUeeNgdXX^!dA=74|X>tmUe;r+fza6PrP|=i)P%0W|7-#mHM+Y}tQLq^4PMQVwT~fotpTRXd22A{g#!y#$v1OdVR{ZngzlBm8u0Mz3m9l&>2RKWR~h?#ep3Jf4H`mL8=kf%-HGs3#atp24l1Z{O!(>rbr#$D9Z9f~U^d+It+?C2s@VK$lmdY&53dj5zS*!bEV!c(?cT_5)Mu!~C2^z<AtAx8Ozv6aW>+{^Z_pU}f;$e;;zgA*0IQL1;Ms96{7DDBtWEt~4}GBlDRK*v*>&_+4ZQbTZ_FO5U-Mp2Oq#nn!K#`b?#)vU;B<eZ8A3_~+50ut8AH;l~v?4bM#OgV<hY<)6AxCpK&8mrZiGoy4LHRg&(v<L&uwd`H~i%5>sNkXF}mkUlK88Cml3iue1!Gla(uPN{)Go*D$X+v!~7s5@-!9CwrH$O^`<3-x>7Z^3;4EuUR(s|eB|!_QhDqTHj?<c41Yx^K5}Gri70g^e2qfOL8XG2fClDiU@>TP~|lb^ZLipvuh7YSL3{W!0^!R2&fT&5DT(W+SJyN$JlnP-{B~pxpCq?!EqfT+^sI2fNC6CnAWH)PuUv6C%PiuH*4+hl7N*e51?V46$icdkk8~dKrxsABs^38m@lfvV(7g`O&9oj8`ht>$~WIL>0(f%42_n?dwI1Gi3ucrnE}ytb)h*v!aE3G3w%%-)bOfK2MtI^MgRMt<o2c{(5dzrp7@|q(lFf>762VuI)h_r@H|aqWjvAJO5=<tBRY~j*D>q!Rt-NqYq^8f5mNYE>#(#r<;5?i7R_VV9cpNz`IsrhvN6nh(l}KU2O;V7XE3~kLFW{hWV%(_+=LXcE@cWF~b6Qz0lg3BM(?s!KZ)6^R?#VWFOe)sNWc@U+Vdx%tu?UdD&lE`R468xwqXWoAjE0z@7h<K*)wkuZecE`EZG^wPy)1B}Hud*Ttv%lDy9E_1J#E%6B|hP6S7)6RYAn7_`5t>Z&3R@auNbt2d4&Ee~gCbD9}wyFQ8*@!&;rlT&NmfS%0?I(%L;`#D%qyG=DczJ}jU58O?ddVlu(N=mv9JCOVxm8g(aQ_v5Vb#DBZyt8RRz63!ya}pj8#bx(5&uZg>4L-Kg`}2-Y%+?)XdNW6DdpzR15IWYeY$9Ps_-w(?t2JZ!Ru3xc$H!5-CPn>z?*o-5yyBb8mcpU-d%NK$uU@K8npq*MGJLQFzU>`W)rUQ3vWuv%&6-=j1zY(})P2cZJ5!2g?pJ#$!>!r}Yb+D`wmuh)TdaPl?8ORw#b2$R&dPdn#=(c+LL6P@=W(*0ucttho{fx)_ZDM+wr)%=^do;WV&XpQ`&MLaJ?aEqiT1jAaKEeB>(j};Tdfhy8p&t$f;Q5OXpUa@SAX0uTMF7-E!gaUC_w{uGBQxO?QMmu?nsrgwwuNn-Y$^Z-(P>0)wkFfG&G5>cbTz$M0@<Q+YYblNieSl4-cK|6xpWijVA;)qqs>nt)|+n;@+FY(XbHOYRIC#ppR{V*4BO-%EU>vY>YeYk6gciU=|UFXF2TELjI)yv0O+>A>4Y4(CvXz{=l}A&cIo$ulygyt@oe!Vg)lR|9-B_q$afW5Sm<E0XtqVk4N=M?bZGG1SjK(HXul=esAoSz9(;Vc<i?y$4G#?lZ7Hr?(|RN6bHv+UxAa0-QDrC>7Q`4uxNR-xa)G3MJz}6|LptG97L<60ik?`4+JlEBR@)RH7@@6EYj0;r!s?`z4|-<cF^EmJVc&{*T3@Rv=809_hQCO)eiGkwjv#y5*0dhgKjM=FB*4`*yr_hEO^|BjTvdV?8ZtTRKQ~S;<Q;(mE%cEYwF19gEP!LD|Wo%RwTHU6twkMwq$RC_EucI)tf1x;&xCrCi@G0%1y_pPHcxleJ_0W^tDCEfsjsoYps8ROMDl>Fjkv?Z00FQ>ynb~tH11Q!uA)fFk6)y^~P~gdn;x&XGhqxen9%LUp<j`=jp<V*1b1(!R<DX($#bC4JuHlIHK!#!&Kl2!zi9h4jWUi`+8i6U!_H^mPTr>&Iz*qZF=bh7;O1^W%_=`Lj<2<j~!M#y5akNK218xle`q(RGqyT|9ZPo9o#~`N`G5XnC9r<vF2L7#G{Pa!Ua(o(V+UqdTEtXVH27`?;eNuRV@hQn&cT0GOmP6rqW1pPX|1xu0KDl(uKYRqurtDxl(W5cz(v!*T+Zm9=K}DE|?B-Vm{eXJU~i`q{?GZZQ1eS;k}#Xlo+X#Usg|t&SlM-F&gbDaUE{>nWfbGaDR3CF`{~%g=`14E3V9b;b3C6HU8AEq98bP)>NIN^lZJ5WAjcEqXqL^R^?TB;!<ZgPR#Jksy$3_Ol)-=m+*Q1?lw=~!c)<>GP!tz&d!+l2fR(ZVz;@-_^KbR2Et7iJNHVJx(xUrC%2oMZ6B`RVRnS7p?54{<@eF|z*U4?bX%s*qtSG>_BpHvh=Po0E!eO~@;p_~5inOFjGqZpJFUVSWC9l{?F7pupepB`Tih2^9OlI$?DwA9vbCP|MQq9RI!weK&9l@m1i^V%ldYKukv3Itrz_Fl37w=DRN^d0i&_9w^k%N-`Co~2PE|J^S)8@N5^H#Yudf6czP1UfzW@3nl;PR>P-nW2+KD3vM{M?5ruvauF8zORMXk>t21B1TR!;U~tG6LqoNkd#!pxehp=<Hby*|1g7a#{~#rT%2=A4u{NCIY@Gm<lRxk`wwwC~S@IuG}K8XSHVYUs+_!aHehl%qp!sz=QRy_DBSz~A(3y<0EP`q)e81s;LHuDWS8(>*L}>8*1g;I{ns558yq@+4;1Wec|3kHsv%66ip>VEjHo3vXfS{C@QZRpHw+@e}$9{clNiz2(UpKRU};BKv{`={n0-ucsMuoq`Bu*Sezjg8`A$>L)xx3T605UB>mS${F@K1cF?=i#aO`n`c?_tz&aPRR4^q@8s6r0lg#Cv_k_XliIO0AU9Z?LmE6Y%4+j|7kKVGPqP-$zMhc2GhN~rq}rh-qdtegR_~HBXX^DFwE_j~A(5#~M=G9DGj6@bgr*X%C~*iLKlP{-&yzJ#pR^)qoKnEY>JA%D<TPPtp*eNn#8Y3}i~MarNE!GCiih7Fs&=xS4UU`z*@*U$M6cSNvb@~`AxG1jI4`&dG^mKZ7hMuLA+Li3-X3Q|>~0YomJWAhYcb-5g#jDrD#trbE|d4+cY&?C`zv~N!$I`}_o#LVyajDq3s>n21Z3khl<SMVH2B`7%<J9lB-ikKc-6-$%Qq$u|9)v~;9xH0HK*w2IdVp(a=ITQTPw(rKT}mz_ROuc^+ufmY55|E$m~tjIYhgc{qt<V<AL+IHHg2f39rEkZJ0OH<c~5ouWnpAvGTk?SmQ$k<~Op<t27Z;2U}{Kjw!MWpWUiVF{$FRXTZ(gxneyWc$W>^^SB8$jM?r~xN6FO3OOSHv`iU7;aBr-%|tI)yNY#ee39OMmqQ}YEx6Gn@N`Ax3e#G4T?aCRgJle+-3MFE2dxDWNFZ1zZ#KK1E1w=8UMgYMg<Uso3q1omcDsK2y*s>L)9WLbj<uz;8SDytEm3bkhomaLOS+%@9HxglUph|ODHn5Z1D^f3uzo*FCMr{e3C3hsn`qcPSTqOpwCw%ZEBdnC$*5}#+=GgV?AY*bGYcW~u2^gCn@@W+$BXs=ky1%9wg;l}tGrRl<l1i-uig0Fg2--@)8(U`dGQy{(`olQCLuUzZP3pj%#f4&RveOfY_}{Unsd9p)%5z^#Rn4_jMj+NDVvE7dkwb}Rb#)A1+~RhepD)rOGu6g;u$U<bHFi9RsWuk0Egh(kz6ic!<uC0z`#CG&r*+;;iz7A*7T@V_m;B10aiZOX58v3JM>#4)EJ@5W*e`-;dpr6Y>93kJx$8}bLFR4h-Bw_NBz|)+Y_d;@3qJdFdDV6w4X<OXK5IUDfs9gz)3!UNX<|+X6xCi)MR__RNcWll!Vnw-~9q)|BYHs2X=B)c1<6-7RT>?ElmzrdGKy_BnuCUrDhHm+`K!@bZX)kqPNMUnK#^B6r6iw36}A*=faqx$@L?6{HG4wFZtNP7Kh6zx0<z!LLJsAcTO%?<IiJn-4pa!pC*ho!$#CzC!Y8C!d=ayQAoA1=4}f8uV%{c)ycoU9|*uxM~Fg@g5v%(7aC9aQuDpLoF+3c1G7PjoY1p(AMd=1z{7yB?#3I_l+i;zx~<G$GpzNr2Wsv#4rp9H%0;i)Q`fbiH(N?a(tq=3im^G9q1``UjSYug-|vrX(d<_kzXYohR(_?3q!8r(Kp0TE22Vv*08Kwd>v>gDm$d~kgj_NlPO__EZ0jFSKH=m3vS{tETdWJDV#^)ae!Fc9&Bl~$1!G}uz%?GO{y1zX(Gvc%-^<<)g=^wV%8E-8uTekO8XM!8e;*&QjIaG&+4U36uBdYTHqdF2sX`Y>mTWt0Tl78)!|r9|eZ&t^>2ujrdx70;J&`8R#sgZ>q>!7|)hycv;mE!o<$C9bUwm>RKhJgc)OlxcUa0pvI<k!Ee4Ci7*QZFkHTwYt@ws(zqD?^eo{ywCoBY~*l}}#dr}$e{RCuu<&KH?hRBhczTbJ6Wl~kw>%g!3tE2rChPDS@boE}KL1NIt6=6W#X$T-nWU3ovAAK(ABFZ`oATA=f%FhpFMcf#@Cgtj?Pb6J=c9tpIZOV4R{bQ8CFLxaGET?O6@d3^6=6Y2hXLO8b2jGVO2>-74E`GU?@ZRB2=<7a?Vb$FR=@fX)XFUMh1x34$sK$uxLVWID9Hh{(1<?49{^yw|+HD<nud-rgp9Z?7Av0x+U!(4p)u-Rgyd{i{qXM8BF!S(f)nSOtH@44`Lq`aSw)pdFm68oz*yt>1~TC87I9yv5!+Yt52X*DFmC2uz$gXC~E8gB@MXTN=!b;HY@>vl*rqhIO1{q9X@+TN`WsKP8C|6}WHo6ZHhJ^Wc3GR2tbL8VEhq=bl2yr4vSKqM;Rv+wJF-+RB??=Le~t97p5ahNry_+3SR)ysK&-qt3gw~ccZg`_3L>+=t3)4WZz<?uF$dVV|OxP1J(pISzxab}gZK{Y?E)oZee)#oc*zRjMB-LU9h1v^?LJauZaVl$ml|6%FhsgNWLxNFR?mbLQe*{*hhJNH6aH>2Y*YGr(ycd0b0(?vROa_MtfF3h+11h!3KE9AS*(6ornR`#x=jlCT#?bE2@j$psh9fCr`o-R57-jTZ()9dY4;$1AKrvUAS<V~FnYR&GIb)2Sk))HtxYJiZ?WkDvG=WLrrlfKYel7EAj$G+L+eG<x===>n|chqrSFbor3l=d?H&_QnFLDkYDV4B6od2&=L?e{v_nvXmHKRO^p%>7(-ktdr>pDo?$SQ1LFzZ?(kCrhKjADG|bS~lI;==lN2>w5I5RibFJPzM9_*3U2Jd=2$YOGS+uQ~S_76pTivK52E)b)#^s^CqqShUz-M@YCI(0jUS8>AQ}CwWb8pg^st%wD$+2H93po&Ga}N!&;{hBqI2|=>s-htCt!Kwkl8k!&>*@!@wr)!U|sHdp$ibI_KDNrUhPaKU+RFUfh8S`4TE!Yxl_S(sn&G!tzr6%!^U`b}HY%OK+Me)e+%m8cURz3Tn_1{{TJYCuy%6_TiZkmv_is^_=-1-B_I|+m#%>yE~7^zd=p!!cZ^1i>+4Jwg+BX$$<QOye|MA89beW?)%RfZ~^cZ#$Crn?J%0l=8>iq9}0wNPUQZXxy??~|1NY>ID!+@e;2w56ro7!|Gik5+n}*iYS(CYfD(BYg5~^ixHrbN*^HM4N$b0i;}+Oo5g#sYGEzjqWe&{eK9N160u;Zr)L;ASdalhc=f?AC8~YTqQBShF;I#hf3_N&HPR@@a0rL_bY!)O_->^mUnFQRRE`D1acLV>@@>jSmcK;IkyVvB_J@#84h_9sPK<<OyW;7M9_mV}jdv`yqojP*^@k=|c@Zc;>`Y4_?w*2)#yGQV^ybOHRf)>km)CBJLFqYN-4n@FiMpvrcRI0saODk<bfV-PJQ!%ZYAUPs>{AIs%u_|q&Gv+TZ1!eA$Q34#*ybCJHF5g(*R3IZ8Hp{!j3B`DW&3I>c+b4CL%0&g&CHLHHUpP_9e|~!X8}8*h(b_4b-SzXUgIb_WJ>s8Q;h^K9T(15;1<?gTm(=eO_w?|1>2|z(x0!ubA;sLs882r<Qhk5Nb>gl1@Agfe!|FLYPNHU|^%ve>7$t!p7Em`~-dfG7vbVYRU9ky7QF9u+OLc367mM#r@SO^i!FFBje!hEvdF|G#X+2mrDqE>|Tl~pK_ZN0@6zo?Pw;RfHzGx@mZt-LOgs##q$*?Crz4+RcljdowHt8Ln|H)MhybI(qG5i!*b1V3eP*A{2ioY{??>l+s&L+C})b=yqPwU{c&9zdy$FZP$pN91dsE1iFkeRx%5noG3c{EyIrE!GJEqxph?D24xFRJy<Fs?T3zaUQF2?eJQZx{MaJx5L1h1=FL*o7Bb*k?%kItS3WhpB_dQ(k{>WW1XXbuE1E`Q;ZX=JN2K0B<?Hq+DC`EEnznsr7J!0G@j?DxJDpj<{Qi-tO+W$@Ic;)<YtT$l0u#Rpong@pf?ZKB~3OUt>0Ss}j4~rsu|YHzaS{jHM(expbe+<8ZKf5=e9<tPl#yUANqd)Fe|1?QbL^t{>dc6+fQVBi&l%mF~LEkHiR9;uT=CI6(_0^J>@~SL`lBsbbyd^;Xk^YDuNHUA`OU4Ih5gbDo|cnW`tR<6}s`#aJ@#cbF^h-1@0tAvE0caIpC`I)(h|)KI+YC)H64tkwb-s1yw8fpkg$TW#OubHcHq$$pj>w?IZsuj^X>zV`R<Lpy3lQdNJcS5B+sgbzx|hZTWWSo;f1^GPqv>jV1G-oWPOCz`lCzbk4{ZJKgA(c2G$Dlbj-8fh01)T8C0Pj=MB--PZ|nzaGkg?C!#`3I1f#F#%kite@^-XtaU2{`z}uYYUYiC03;cCS0*f64k1R&>5Q9&4X+7k7dFIJc@`mtqokCUfC2-?R+`eiLO4G#%;Q&zF5HxPiYPE4+$JcXm|t;N+Vl6{CJA+gbUDv=5&1!#oqp{_EWj=;31Jz`N{u+T3<^`1h#oho5Q>-(Mo}fept`hE>!z=)GPH1FRuo`1Qh|5_f>wWLz4qZ%?(<o^Ck&c@UHEw=E)=ilf-Ly?YF@Y{x`UjRbK|zYVd!Y8U$IYCve~ga^A#wmZ2~w_R71kAtzV(Hka>`7c)&%8s04XmKq$X>~XaE~3v2fR<uKYyHl9OQX*rZE5F__^(zgNat5Q&us8V{kzV*QWd_VDBs?;l`%Vj(HpT%1hmB*ZzO>^t;#kWjqIJT2I<Jviy{|TJ9`y(Y=+9vsJCHTkbVGD{txTc)(+|B3e<u5R_6~$-E~!Us>^=)10p|NOs-Mg8TL4<D}BSw5P>!kRYlOp0Re9m&&9=W8Ma5yyH+EFt2u+F<)~}nE-BS-gSYggw`*f)RrFZqB%1v|)l%;)<4Ln$=`}t$O`t#}wyX9Nq=;HreFYxnz9{OkC|e=#fT?=xh3OJGOvNnz{0wrwTFNik9!qs_M!(?9nNr(>`04<cwKZu=@VgKF;%fRwdbR21`MbW+N$=(#Co<4`{^b1DSbD0nra$AE$B5MaxVqbedgh}L*HyJ-+IvbH&AU{ji6)>v(J_j(`Dj)36F!7sw4*g+Z^t2!bXDBdwUyW0enVb&TQ3}m+jXScR0C<hCmYBb`aQnETir6?srGOUN0Ljgd1f9JaM>NqL=DI(+=60a{Pf>)dmp~Rj(g9el|m;cag1U6u&2vdbzPVCZLxh=wi<A4w^U-rvatd1Jtujyf5z&YcFR^CY}9DtGGIacRY{;A2N0S2se2&BtI7Kj2KOf#OsoCdES?SiB=|SQO6~j{A|2{cJ?;dZv?CiFZEu#R41H(71dUdG%%XD{YlNLovOD)0#P=A7o%Z~i-$QuSVoqy!G3*HhcF>8NGL6?QVv$uW{7D5DTee!^P4MCROnQb>HxrF#!2I?-k*hJzWm!k(W%rbUr}l3$UBKlGr0>?Y)S|S`Iile{RU&#};kz_G%{ZQxbE}&U^WZ#O*veED43nU$6?;{=PU9+BSE=dlTXx#|_wU)~se8=Y=!H9~cgU%6kVy_&4>D`rDTvpI1iyxdS-TyuSledF{_mja-AesuONeej-5({^Jhg40ub$g*hK*SzrS_wtP(rVsV5{9)WyKt8L;1TsUreNXg=Pqw4=E<RgvI2WvOu}L#=Ja?b!}C3U6hFD?5<h2W8kBr%QTZ)&S=jH(QgDiz~dHk+LffTtR>;bL!KSFcNOQ}NwK5;Cq4z?7NxUCq=(&es?`{7ZGl=}RJa*>eD|rLnp4X~-&g}(#^fe`1;QeRS=(aw&wg&$&%yFoTEgElY>eC2<+;|6)}2~1Oe<p7#Ruz>>MwUx(m$)OS2UX-2sLj4b5eQc`6>VK<+;>HhnTjRg~7AGyG-N1%Cx=hv$!0X0l2O(b8R|qU_RtA_L8r*;MN7Ohv)64_1W`7kBOQyZf<G%+|zlqdwWp8vN_wT7T=tzd{aP)wxBHQcX78^0=<X)B~hu~f>FwIYPz6h+wl*KYY$#L%oEz{H0r-*uzFCgjNa*Py2Uykl-B0BWsDT&rUC=4BaCJA)4jbKu(f8ho>fC`+N#n<jJmOU28Ra<UUgF6-mqgfSE7N?Tf9bK%qc!Fe#OBOc@QL;w5vMM9#y+Q<t@`y1Z1v=WLMCsq^}toS$<#djbL%RxPvwBv~*<F@T862I^Rmws8?<aW32%lbwV+9z5JH`y+5~pM+oIyT%Y`%NaYKMg>>!Crf>sxHX2TvRNC-B4gQKVRvQE5UCaI)blY?mdwK~4Xiey4sm#JsfR}?4bG@|x6kNs?LyK1>!@oflu1Dn+%O>~Xv$AmIf!ae>lH^~R&EA4KxU||@afvVBns!S*#6D3$tx|DC<a~JTz=l~)(9_x`fLWp(hGgrnwH9Zo(s2qBTeMU)Sorw`4CbTrsUNiFQ?Hr@-P~k$k%*L5a6d1@%k<TJoOXZFdL6p?w0Fu#fBxIR6%!ZOjeBX8p9{S`?fgx4d~$^mh(7M@N7!URIoSN##OnKThd1~cREf8TUKqUak^eVxd#lfa9Vl;ajS1E1Su_m_L|0|sy6)b@>rKucgM7KzPjjI1u@qwrfyl#mkLVUhhN2D)7R_wac_e`Ni+yWDbhp5D{v_IeLE16QTlyzPUp>p@xXe~@mmlov(<)^W$3N`;^+(g8UQLM^nMt%ebiuI0Z-&cD0hssUczjT;u`i7D*;gH!UDg@vidCn<+u9))&|WsbhuQQ3B(poQuzqAs;M>9O;KCyj&oL;Bn#0O53j%gp1JdLardx@4-kMJwdf4}4-20FW{#YBX`e^Qr<6(YUssk(i7I|Dex~Z1mr=O8-o1kB1GkugSXF=$x`?l$Q+wp_70QX+<%R(#1*L{!V?+bFPEBzon3<T51_)f(bZhFcjR-5~+u+|oh0getGg}g;5Rrxt4{7_|kKE6F07$W4=DKr?)1_axyO-I1@tv7myx}OZ1PJL174N!6yK+XHFM}F8{wB23K7kT*swWXdlap|F$xQVnxlf~j<zNJRUVRUNsMlU$Q<7f`rojnBeDU}UEWw#-3C@^onue%pY1Z+L4ZD$8C&}Ow@6-Lr=3MV(-JFQ;=^69=B@)@*au5FsH%H#a~ZV{xpUYT@W@9aD7#p~4h-3PxXd{}?6`{q{~H_m^>KAo3Tc@_3A+{fVlwi^Q{mhJnN91kNPDw?6t9$Es?{8Y#98-(ey?P)Lzgb*x1lVMbu)88&|5Il7huGOmej{6UBkA2T~F~OZA`o^F4S50=-)>Wr1o*mKZFFBF%nuy?d8~4^ZIC`Qh-`E#gq3DaE(1$g=;Bopr^}cB^4P=WW&V$k!Eqp{<lxCrg#_^)E@!m_$Ua~WDb{}k@OGnVfrLg$2C-i5z?MHJ`>lR7c9#4SuG&8DuaeSG7<d(1`wbP~yqVxN<0FcW-U*;_M!Z@24zWVZjwICYl^l(~NS>*~-&>Yy{?U*}_5s&&C@5QV3+D_^7na4gz(;kKD=)G*R4gGY@o|~NLuan!y!|r<$i<bC4yvj=eFFhWP_N~fmNk4fUFITE~ojo3TPenm6TC$S`w7nuZv~i_j{Z`|YNzzHSzjx8A&Jv*WWTIkH<UHhv*b8<=h*3uPfX>8)%aP=qew@?C2FCB(mUQxpab4NKo$G^J$0R@xorK+~r_s2#{=I9e2t?WVYM}M|;`lnrzR<<V<CPuX)6TcX)_a8D(%E<2t4{XB=tOR$#{5ORM+GuA?Poz%7WG}nT}1T-u~Im*T7Ne#YjJ9A=E}acIdzqyDQ~NP`RvVgk`VH;wR-hY4yV15f)3kvbyMw(SB~GI4?7JiYJFw1o*Q_}I_BvNOFgM2Stet)ptxHFP33~yVu(aXc;0zS?am~oy05Q-Bxa~pgrPzaYSC|w32Rkdr*ltael^WWyMeoVPw)}!)Hc87S(rx-URj+Nm29!R5!1sO9ZLH9O^l{*i9JB`xfCTSMAQd&@%yn3DXUIa;z%dGkvnuS?zgTG%j%>FU*r37Dc$<!s;VNxD$;*5yCkdSn{cCH8&ntm9?P!|dUPA@zT2J<70GCoUz+e*eS6*R;d|+L*t0!0J6U6qdyUidfTg^4PUUMin<B$qucfE?c60=1kD34Ex%uS;ji-D?(g(#)>Z;U2od@%1w@?JSdsr-NfESVhrjG<z$h9|?^#)@nuxbCXaUMUvVLHeC*OaguRd~>T($eoXQF#shy;ujS2v_@;q4lVp^g<7M=&=fUJMci^BeC`>&S#Q6B5m+Mir7X>+56#rBk;4h?p?!1S=m`J$Etv%oY1YOwVZBTko5C&n~liPVE4K0MUZ)oO^=;5my6&1t%0F+09pXPsqOl~`wbG7>cM<SbiY+?ZeO2&zIi#+?Q<TWS03&i-d2O}i`O0nW4*h*46o(dJ%aE}TF8pA?{X*eve<4PD=q<u=0I8CB}o{NPvfXK2aXTl`aygVdux5!M;2_CY&e}n+_2`NHK*>o{>yJ?k$5)I`{Q0avJ-i~%zyYxth8IJgOzg)Zz+?=!#I(YTH`y^^PCgu@Ijw~xpJ|=Ee^8Bcq&z0qGXAZOR2$mKZz{0Y4evwyLKDUSFGJ`?VRnsYgT*N0s^aQGB&FVnMMp|F&yRizHjBsBX`-;?j^lSX@!1C*mN!)|GI+0q_8elHyW!;+HG)Oqm_K3`WpXijr;k47<hAne3I2#&GRLP@AKD0?<k<_*Qism0Iuk=g3l+ra?}6wN~o|8KNHy?kd0D8W|QW4G$}hn6FDv_=*ON+&XbUQnZuhIPsGo(yCctAn=d4;n;%x&-}ri|5OQ5Ihr`?cdFsHxhrp9|!BCeA@O=<@i*^FBlHTvOu*L&=Z<)ZaJ90Ah&_8N+!KP=ma|6e!RlnX1_Y;)}v0!%jVp79nI8r<zyT=Ph`t?eznx?I<qp{br>l=nOGpzP}{Sv?GiUK9epA5c-M@K0A=lgtQ-K{%*$-}YY+aWXfsGY%VpS8P=>Dk}=W^w8qPH@w6-qc>49<SMzxk|yH!FhdpiBj?fH}oewbf{=s3}L+A^#uu?{C2ieXU9LottU0`FS?`s4J3L}jZ*l9vD{)&dVkL9G52_Wy)B5_ATlimz}EchDV{HK8<^va*l8TB-Y;yo|DIXFh^s6|YiukoXOBr(D|Xv$=iYc*PXK{nX(9GrP5n>WqUmV6EjC*(J=eHg3!}0QA;CT^AyeLOd8kznE8$RDRX`fGzw6(vCm9DuO#9k{OFHj6<v9E6_dM%)K+k@r;YM#C=oTpv@T%INai=>)?d-;`rwH-&E9cI54)dMs#nAmG@a+~u^s{=6XQu}2Ym7FfE6H|ddhb}+m`9=Z1F;(<4}`?8Pd<_R<;k6Cdxs2Oj5uQao4w&=nDu*g`Gd`xmE#btQn#$ZiI3G$VP`9;et0e`;=UWr8gjTY>Q-H#5aEc<c1vJ06@oCY4}WUo+Zjyr`4Y;-9k<3dgd0+uBX*q&wxn6f+F1Gga5!;d)#^*oX7J9;o-?TSLU6SxcBAZ?i4inD$?QVFJ9EWq=;%=!#>^XEtd8_$+UbYp=moB<f?D82aePvfQh~x6QStz|-ZI8~92v+%@OIt5R5f0}2Pw}3akKVz6M5XMEYK@P4v-vKn~lzWg3OD#RcQ4h$uc#MvNgZN8;@>}xu@y(ZaX<$_UloF{QS(lv+W1Y&_>Ssp*<URpKEzr3AO{(N32ML#|N^}GM&33YRNdecH&zfz3(prgv^Lblc61#N@~yZ;kGX)ZFTDnGG=t?v<E68@p654b`RSsAu{JxrLNc3Wfe%VWGI?@eYo%Y=TFfi+D2~pNN&02sZw&)pV@b>T2g#7W5~xd*pZ#A3-O3^m@y3k$BoPDEcU$aDr6Bp+L7ecZRI0!T13)|p5H8wmuz(*+%)0wY>ImH+7;3o8x;6C%(*jQGgGj<&`8=|^@V7iLc{%U?Gy*>JR7q=y@JUGfHX|@o6eU*=&!``;5EJN_(t^y8Ej_$g>uy`|6JxrfB88N9~lUrZW@&edh_OM>jo*;g~g>a@LE2S);)NQKgut-)e$9FZ!f^P#yDSCeV4LXsn0$TI)9=~%4$M~ADGYExAy1fii_yuXxsKj#CN^d9GOMkQtn%axrjz+)%*UIo?zD?1Zfxtkv6#a0AlSc?n?c7@?|Tb07ii~TKOcnrrJFBIQNMfR%fY3f;Ozw<ZjA4T{KRWgWKr?t*TLR`u39G2u_)W_%t<W^5@n{?Csvr%V>TTbCT<`ceVA}DOh%L;cqUqwSyL|lu^T$fyYCKFpvi8mhfyf(Nfq2P4P9s+v9=pc9@Rmq5I9SA?)MPQ+{xFkBR*L3ts9su~Cr@;;iPOec9<w<Ow$>8mQ&KMp2WhR>#G64vpQ#BYB>uhoaX#CGPM}u-#_Uc6+TBY#Seqd{4{bEBWFGl2!uujxYBU=cDqN$aQz!7S<~A@7A^5mA{vNEq)`7ho^&Gy*8w_fZzKYTxq9%7a?za$F`%*N2>Ie#@>`Ajb3o}7GbsgxKR;6n~z?jm*QfVMzulQM+KN{;AX;7(~$3am?ijUeKDI4pX<$NwU=#`Sk<=Vq^%76Sh;?&4Le^wTh({J!X~Nr>gK+FVp|r7Rez4Y{A8V&(V`!wXR7Qh&i>p4Rx9`h+JDHW!SdiZ;g^rF+%Hc!H5jinbvpxQUh`$nj6oJWmGop=ET@kXZzf+?SMQ2I)jP&*wqs_c_jZrvd=4{63!UAWx6xl7ryw|fay3(W#*ol>eePYmRk7Z~dDrHbD~@bY2f9WwB*}^mMYfM*abtG&8=KJ#?isnzXVu2Ap4Rwy32=J6*>bd7UQ)Yz0VU8Hx%b9V7WaB<?~ur`aqJ`2=b#St)k**pHT41ScVjc#gbSzu+~;N=zo46amNXtJ)j!_R<SF~w1=%IRnQ8GDR63l^6j^;aJ45^d>9<e!`QtrJ$zCXpwd<KUk%6eSOsamGDq+`7C>8sIwedWCjC<zd`&E7TrX3iFQ1jV+7~#-rbi4s4R0y#Bb!0NbGjRjC+%dAh9_qKZ+iJV(@tyXMJY~5I3hC!@%HB@d+l}LlfJ1asTPm5_Nq^r<>#Ox7u@HZX27wD-{Ehdl7`U}{8yQj~+bD!Xz%{Flb$0_Kw<9@kCynJ;pNu!Z_f!5&Z3=*xDHk$}*QBv{!6;>-?fou$8nchWV2nO!xz=$Ey#^}3lOgx$OlHQ-37qyX{2@y}4)=L{KE{^V-eK2zBT}f_uggWMda9nd*3x`#Gh)5>?*93ZV%p&K_~zQG*oO~{>fWmkGsDAUD4W&vPnPmodTpVdw7)=R^imNrXNb3pw=`Yk52Z!3%kg2_v0;_0hqI!WjK4s3TiAgX<H}=p?R(qpW&EIfTE|os!XFA_KB+t9@bt67tSd@S?Ea+ZwUfNwgP1Fbs2UE6Q|80$4B8vkmY73Yu_zg0By@KRdXF@#DnM;qdO7=q%e>XN^ZVP8@XoHk%DNT1zc$3g#-i)*xz8qJaM=5ISGVHH@wYS-Rb^npwK;m}7J4oJBM|I+$$Uk($58cCQlUHN`#+6x!VSYN*y%2TCvWpkCMO*{@<BkvhLz2!9$L)1mDU~hC$YGC%fSPBjt8kJUOTVlG74Xl#jJwVwujwpzecX~y0+?QL9Koinp85O-MJ$?-t~z-X3wv*e5ucx&s)ShfOp6Z%9j1vbl+GT$0r=}^U+=#GsIyszu?^cyr;7(-I={>FY|rC_1dxaSiJXu{)bpE@9Mw}%&Qf(_&y|=)ub%3##QEpU34?ItJ+Ma8v6Gf1q}eJQ~g+>O)flWGm(>9sJ%(w^=ESBc@Cv-54VL2&F!>)>51fhh5kjn-2mEXoT=EUAw(Tbv9`)+&N*M80jKmQJ|a>OO+uf)zS2(G=@Fcbn2j~@7)6Y2qtI_tVygg2KbC%8yf*!oxzM<^!*=rdrFlP9mkeq*&R_Bp$w^Zq>cOY)RX6Bl2#yw`w{pq-zE$#%N9`I-)~o1`c7VP7@WJB7(WuZ^6$`A7>aPNvzN!$n;pXFj`6<~x-#pAVx9EFsGL+Zb^y3s*OklsuN*{A7rZVWq>l9dReuGgS|3O5O*H#Mx{>1$AiPl|bBK$Q*W7^j~v)eWX+!Ah|eVPS_qvdoTX689Tq$Nj}tGg1zMjtYWAC5h8TH&<ggmiQZ$YKrz7URR=N{4#4p&?$re^%ZARoRDo)D!ZJuEUE4L#o5qI2g{1dHwId2EksLmqP2*N=<_2XU7Slb?+~#M?RX$r5QbzGpGB?cM~j%lnPiKh)ddNjJ$eYU4ItC4}bj*mR-(O?z7mK&hI~BZ#w*EcHEABTG0MB2Ta<nCGBNdm)~$>hTmN&r)tY5jJU)6dfl~wc&zQTnCm(u7`CVOFx*#5W!yqJhz0-+<Jv93`J2nf+T{v7tQ;oBlYwsB(m4*&TG%>yE;Ndek}!-gw-9|pL+2YGVkFW#-Wvd1Up7{!he59nlVq&@sQF#kwswRTxNg#i*51i#uGG1q%TC%_oNxHAk%Wy80*a&Qs%F5AQ%lmhn!4&u*XsFE+0Uio*FOo#f};n^l{o6KZ6{L3O6^_$!H>felJn0iMTbMvL6f*}P8-XZQCWO?@&b<tk8<jX(fX5+6Vy6lE=iVy2DYkxtRdaG7^_tCf%TM1GOF3X*ZYinnB9-}DZI)T>M3qV?=CbaMj+hDCZqF&))l*Z^Orjjn32_5aPTh1hz*xu9u?53#U?#1ZfFf4Mss|6u78xKDKWg+%9jlo`h|-a_qsQW8-*B<6SS*A>iaYru0gzW$mwfzkedz~rnGUC5~{Y@n}4}kOWpQFnoWH5^xJgY#naoogP_{3FWfD=)WX*_X_Q6pAhy9%Uu}*@&6t#o=&*-2?o<q5x<3`?&oy7zvvqo72QH~6^M|V{Ea^AYO_hC%QgQu_H8#8|@c3rm8KDucP7o;{<|HxXr;|0;(%4VW+l1$;#CghM=p$9;ubL47Zl%w%<zJ4oSv{JS^T<0_Rk?wjHl>s_+d5e`db|E$HIJ=QxMa<RAh~a4Cog$_d_}d+E)-EPcxgfNZRwRK2cIWb|D=3cL7hK+C+)y(o1(*hw}3EUI*uzyt90NW^TP(&@3%?WMv(S(y>69v?zCX^!)~x%Fww|+XB%Xet>uSK`dZt5u1;fdY;`+(D*Q`^_7kZ)alUh2z42A<cCOz_V#uSFRJ1ld^RX_$HZzUpLi0T8Uy9SDTSied(NBSSb^qS+a0H-(nQ3+HRWx{1IK@R#s$P3Yw~0OnHO|zSO?p=3+oAJqH{R7Im1Pi}RfNh`pHPhdNi$~4KiTQLILX~}{JQ4Bf}gJwOf9zU5q~IRqkKaMe~fW;?e*%}pAdOacB~5T%bx;#k6vDgDg6>ZNu%IBPnjH!ZHc&8=bpQRxQ@1FE}p#m`B(RGkd{-skHER6H|t09d6E%??Kn5()Lg$4hN>d3E6_WmNkm>4Rx!(G-*})_UHhnF@@#)Ko-w{*@H|cQA8)J*uwjf}6M4z*Ev;*+E2iEgS4UEh{-6}aT2+sefbMs>Mdx>D`*}8@tksvyr8}Q4N0W_HI-@k4np%`=NV)1{hnClMYMZb3z%QF>VST*DokcmaAi0cugDrI9H?Rtm(~a)KF~y7I_8VFf9VW}p37>D4)TG}C#zVm)a_8>cr_;w94T#_-bw$QPH@mde0v9Ljb+=^M^%sx;+`Hnhho1^zy{bHx)}eA8R_sxg9WD0A7H!(?QH7YcLlo0KIZ7R=ZnY*i;%qNz7>|0Xz%=mLMh^POWy+1+LHRfs>%malR>hpYlxP&FN?4L%=;d?<kxBj8b4T*N7U>gf9S;UO`Gi!1G&y8ay|<hR=>5(T24x9{F)f0+R^Abr*ug6*Yn;yvNSXKbcT*d4$}yDMtE9n(D`2yn<9vSQD7lz1+)a|Fy>RPT-sxH=G!LnUdz%V0!no|<5Nlb#Er)KO%Md`8fxg8TWTsPWh@VkQ+&xFHX!oO18gknhNxx;UqNtZ4Dt$y|uXe<OZ8xIEfN=)yRK30U6{zsWsW^8P_Hs3SyPYKaC2MVz%L1{sP_6rJ8lUE)pVmgteO2yX2HUl|QRiQiPTiY&yt%!)HZ}PvCmxvPYjC*J<M6s?Di?qm<MTG7-UHGlm;EzvEX+3gH(HCGPoVnfh^@Bq$Rf^IjdRM+MV&*qR1*#rUxI6~vH4h=l~FD>fuW9XryDh$xMfeB#<yJ)NyhYRE%i?ir4D?MoB!^QsQvDqvF8%%EGT<>%99rC{ozw`{h{8@?Q^d<=TBU%8b@&T2wQPCY81nTkh$h;rJQcB9v{V%uk1asvkwon-bR)2bu-dXwUqt$>%4<?Q(5#vedNuP9SrsY@n<7{co)6RY4Labrrvgjlh5>|l0&bQPxo(yX;iRmT^u-IvdRZV%tLFyZNuk3x4c^XXfuIRXx?X<7~(7v96np8Q#-hi27GxSdm6RR=YZSV%|YYa!&v8P29k7$z~nlWaL%zrWj%3lloR;_jgV@TSLff0Qu*lEy~T;Kdexd4u~;m~$X%dK9YCL?0>K&IhVuprZ?ACiXcGOt59t@fSmek4#aT<(C`oSurVcnL`&8F`JN%3WN23x&-NTxhJgs!nje6tWW`xil|2kZe$+TB&Pb4~NjE;ZSQh*G4nYMb>>M4Fc8+vvtr_FQS(4=FmU-53Q(V>pkg5n#iLyimYK0uh~>uWU}VM_ijrHP6C>iE2xNj;Vxe4z1?{`o<rf98>`kDm_sc3d6pjz7p@mecu#d+%Rr>Zwphvhw;f{QHxI4t3fy%^$+u+0!1j=O>uro|B_6Zw_YrK@;rzef9+Y%`Zi79MX}=JBufE0Ymf7QsHYKsI<nFSLFsn;!V`t>ki`@2L16ALi$WQ*dN9(V?6WS=P??*`z7T=?n)k3-$ZZsEN?eo=X4smny~|U@zaOJVk>@Iuc7(?sn56XAt8BVh(#r8*SGS)naxQ3(cZ|!B4Cxy!lk-(){vt6PB%vN`U!^TN@f%L;br#DiHEy>xc3Om`2Kox(=e(1ogInMalT(zaEk1y_98Fo>W@Wdt5$vR#U<wwgcRfTc4sjAlcu0~b!VIP^dZ|F=;CP(-}{Ycw{vTTZUdx4g?K+ucEe~hKKgZUFXV?y)*vW(#`cLZMqu0m-&g9>X4LA$^VyU3X0AVgfDa>@15xcQ(!+6G>HFwnwnl6Yqg$i7Hn#V3W;^{e=&n2%4ZOE*q~Wogf*r!TqphjDTTS!z$J}h>LvP;Ote1b*2e0qAtz5eX=UdDN)Co*melK19?9G3PNDfje3~#DVl>=*E<1VY=MQ*>=d(FL>^ORh}Dm1>$7ZJ|=JhVEit>I?b2Yb6RQ;E<Sih>k${vs8gKwsBA)jr$BbbBIFXndddr`TqE!yT}evHUYdpLnYFfz^JD7y#bn(I!8fHOBmeBG8beg}+y6T+Yh6?!oLzx{QA)z|@+mYW)g*jAR&oWRUhF8@VZcFG}LZV`6c_+vk~R#6>S(t;m+<n(y>QHsxs+aV-k>_G0tce-l#D*(g<-q0Td6ciObOzXnCGt3;n5Ptzq?P9_rB?UTyR`GMr42iLh%f2B-TuTWHx9aj`a?8tTH)~d9F^n34p;ElGl>(SAf=Z;(b$`35nsa>TLF!?AJYF?irEUkZl31GG`ipCZ}?uTZKyG(rODzx1FP>7;jXN7Nkr|cnS{z{ZJ;zm*V#=oEWCsV)6PV@DeQg3`fEpLU>E@9UYv9sy)VaxleSD@mys?~W%o2<!}JaMVS_{3NH0+}~HeTI`}wWvWjm0a>-F55<=KO#i|19t!Pzt`hgb%y7>#t$%_njHI&L(h+rphkblI*rW#cTu095VG~3MSY5Hw<xmx|1RoJ6_Wn9sDIf-cPP<c=;Qn}NPA9zyZNnuOsc@rZO(opS#j^>&hMTIbc*8b7BkhynK5#o<$bDdzouBNh67hFw?OkyCV0;+?OMQl;$-@$HO;_%t9g91pZ)G(s{&Dl<taq?XwTvFqY1$32$#QFsu9|`!~VJaFwoNuEAOm(I#_sno9%8tZo3rG<!YU5j{N*H-K}0Hp*iq17$$n9J_`3I`ljJVcThFH=jR&)D>m|J_&4(uvY%1aA?T-Qs*NoAo12j-_B~M|LHIOcKe~_9Vz<)?VB6T(7wwB>vSU76GJNCOWyssMq~pVWcK~m$1mu<+`!<hES;kZzy;pveT`H9gdARub6I`X|A$<%@-P|5$JCIx9bochHts^OwCT?+#{C!*^-}TaEyedv~oM_^8m86uyswdWw$e2T>$Yfh*xmA8BrF-KZcV+IaThP$`0{JSNp4%)`)z3S;LkN|a^3Sh>ep|G%b=STKRewLpe~pu$KB?H6e>(o>y{(DQK3!W1mG^l{!@I$_cHh+M^T(dKjDfOyQU~QOHB~*SM+{?s&qdP)tyFAdgV<39g_Y^Q@kMv*H?j{@9{WoB!5ZVM@Zx%m|2#V30aJHieVRNPuj%+Z?ZVq=w~c?FcAp<k>p=+y*mXeo!eaD3n(EJh^xeRHv7N^TAJqwfw|uHvl`bEt*gUP;1W(=_=oSA_0V+owD@=0hJIHGq<Lqw{rIwlFPtoFq9zOv6+)lb+t1H|q)iQD1j_+M9`<++KE^+T?dP(z760MH(Dp;xAI2%}U!2~<m9r($ZGu4!*f7VP14Hw#3eSh-Nd*;PaJ_WijffNth>sLKb#`kb`Y2J^npY5V<X2(8ot1mZI@%KR|<jo$Iy0y?VMNn~ceuE1)lAjI5BzUNEkn{14+toz-8^#;Bj1xtYXMK2k*|*aEa8d2r>xyLbetg<{b}(K%$OmoiPng|fOa#b!Nq(KG0rqaFvI+c6K)+tA!9Js(2;f_e^^*;N?c-BI9+m%A_QC(G?EkU0u;Xj>DctDPg*V-FpRlbRTXxNAI2(3#F(H%gMo)mQjtr^XuW)dZ%+&K-=G`Rigz#W{B`?Echl2m~$v0a4Dirtov@u2}?3b24%$foQoq)aY>7)6grn6XJe^@Cb`)|)24}JOm??OMZX|f;j_BmDms0F-w2Qbg=i`i^s)h(n?T}wK0J2#i#9N@PPeO4c$_IdEjuW#&f=b;fA`>(Owlv@KeOGg4a`#h^_YZ-m%0+_CP{kEo^=USdOjVY~b8b(Y%jRInoK~xr*amM=WagB{efHb>nbNZ`|UzfB!RYoKeq{HW^*?+!5fCxrAKH`Mz^1gvJbIIO&IQi!$c^k2Oso&dn%+_A3Iep;~$_*aAfav`Dwc3d$KN_P$vjakvWH@t>&W6&gd51Ho<tI~njEZ^3l?=arE3#PKOiD~EPD4c_ex9C0Z?)-Ff;iiD$&ehEt7{dh-zx}m#2WKk;?#zhs?%j0)Q>RY4oqIoWw#ood!z_`Z4ywwx;?U{Z>LsZ(e8?>Gn=R5V<a4-G26RO3Ad(t4|H}~4i9)acvJJK;~hiF!3JGYGpw2Oi}!cNKnznpiz$4<!LN209Ih|cWvJZ(AXp<d8>C>~R<k}=cduDLxR{3(@^ahF+UlJ#KZEGo3;MgdAT#fL9Z^r2h}ISp_wC*B+Umvjt=oOz2U**k$JOTp0>4!D-UtTd0puA&0CZ&6C+HU#)@9_;bg(8;y*yf5C)9g?Tq$qx^Lv5zhuuD5%%a8`nB`cNTs*JL8AYI8>#s`ws>6~1HMM1u)jF|*yV~@P{Jf2hzSR(};G597#>QjE<K34MW9nf#xnu7H+K)lAp;*Y5Xe${2iqyIl*h_DK_IiWH58_aGa?;k^U&p?CI2}tI6G>Jn;z>KKwx-^OYwmCJR6pbIRlN)PoP8DYS>0Xl{f1b`1J<)T7+o9HW;0lw#cS!f5ue&i(woC=JoNV@kw078`A}&$X2f4<7#!Zj*VhRjlN$GtX2RFqd>HJpF=y<bx?{Cc+^yH`fxcV>8w-?L-6MCV!+l^BeeRhEhBicT@p^)1Wh&!uerj)Af9RJLxJO?YwS@-7?`}1PEF6WKaCb@e7d-OhD1S*OtjoWcUu~TA7v~N?@IQ<!fQMbq+FX`&l-0lA7}e<l7SegMhgQiibzT)Sal(v?`RdObD*W>!wyp!%do|m~YU?thv(sk1cqZ4O`zx72%Ubg`XfKp4^9fBq+e*UhaI@ak)|Q_-e@AyCR($zaeJ6ocTMpR17=)`sd0c5N0Bat-*(VZ9sPzKa>Z8n{LldfhumH58T{Z`|=O>UCIqmhHKQ3)6{RacygLzA2=gw}Nq~uxJB;hrSwDo*9wZ<20`5`}sdig!~)4{qi{%g=dn9Y**s=~T2^S#iqMlu`Pbg?q?qnJnKWB7XqbadOyrU9}1am7Ayl?b&@&D}0$HwR?Mn#6NSGaM>4SuY>7?}U-8R-J-@*6=M}G!X1k7LV(x?q37oOXSq!)`xnLQH>XyuM?NpNha&w!)~uXyYpZ^ArtX-@t5**zjy;(OS)gT@bUau;K_yk?ZJ(31I(B{CAzC>2&n8$ANMX*{xJG@IR>q%6OKbWw9~1{2;Z*F>9e!$w*vF9hOIuT-&tuV0?**6n?~*W{vENwz;2x#Y?kwKe@uI)$cgez49@nMu{`2asyc#sz*^QnQnR7~yKQ~2q=-S$X4C?Ho%ht`IoSy^)HrjsDh0GkT|0H0#t@4iar5}ht$1_j2<4Y@B7d+<1$%PrhL@e$Tue7ll-yv-S?NzRfga|?Y0x+Ts9>l;daGhttZZLj<SdWDdHYYSc8zCTwO%Sz+5*_#;Hw-tW~-eBzR^dlLVIiEk|VmorsmMM)&Y`<s@Ee;Q9It$BNZ{A-nzHV&ZfEaLE`fLb>{BqgrQ{5aC8(!#`DSWSu|@NQB;<+K#aldQM7lGk7jK`Dj&~jOJ#5+n?!Ij$qnx0cncTsTcH)Z(lM|VPaW(kOuj}cxm+ol?5bFeC%5x3xRPujg}=!<n&;U!ET>^mF%Cl7;}L`T<xlO<sei@jDl)dvE%(o__v)p+wgt_)ruyzqn0hf0E=>Eoy1ph!phP}XHYQ54>aFtP+4cGFvk#v00d=^6S;ha3HObQFUSKv=T6WZ`tQX9r$em@)G8(RyM#IzUx%Y+VlH?~5FYA(tnRG#pn_`^w+b_o)6=$$AH4FBdQg24G%O~D>6T$a!Z?qp{I#%*$0fXi6DcFaYQ}N;|m_;aN-~&6?FN-}CDSu$HZ@sjmWuL+hr2$H=1Mp&}ObrLa&zz1lx$_|__tJ&Op%C4bi?mgjmSCeRC*$6T0vEuXQKO`>5P?U%DxN>RDjn_&0>^JVRA6g}+<~R;JCo;mqDk=X*>u<L1YfNBukUrM=)QBn(QjaA2G8x~+dah_CWWqBlWr&2o?24tJ2`Z=N08zUlJ&hN0J?nA`xCI79@n>cy&FF^X|>$F3`}Tb+wdb;)as3!ihueDy-ymdz`0Uea?$R!)1XR!w_eqfohxf_U#MR-G{137Dl~78^L2Im3c)>9-`^vMc{BJu8TeaeENwu;ZUD*Z#~9mv{`9Qd>^hPbP2B`AR;u+T6UqTPLS26GLdSAGQuL~_Kurs<?rn|F03AxSC=K@9XF|VfsFVvN;2mAZVt<uFYMaXSK3b(?#?9DPu%Bo%2yWI-{^YPdtN*vlFKE%xqgyjzQGG1;Hqzbop*i)EB7y~4zrX%}z~q@HhyO0OAP#&dvbOD6ZFI2Dd%mk~-gUfmH^gannB?Uwft%K%vpfxozt<HiYH@{Z`t)fn-Z<8h2f6&2(r&mZ*2Zdpg~}9aplg_|SD4s2$CZb1et+TxEF%{C>1)Cv`je?@sM_mV_2fJM)7h6TA*(EF-k+z~9K_|YjSy7SR#YYh6=hIRI$8t;1wm#+9G<=X*NKkUeRE=WUy+enLuO?S`4v@BU=gR%1?&wbb{+2vcx|TckcNCIf;;|Jj*aWi?6iir1Nr#H>pd@IOY!Ws6!-7P`SF;=0C->U<W8}#1XeYj3tkzWN45ImPH|uL`fB|~^60XBA<VvVCFJG!lEvU$krL}zVY+%&)uw|C(U)?~c{x8A*P*u4UMm@snkk2qNSU9Hrwe3O9{c^YkdtN?sfn%1JWRF91sk-H_iX!~PAz|$K_7Ggjij44uXq#cPItBovHa9~uR7Xo2(W*_Z2{l3h21Ku$R;0Oz4=YftRX_wW>aSDAzr(@yTOaD*tC59ln!2GT!-LkXxoU(k7t{5Yn7hLR3M{ZuOgSn+NU;lFSdE&)K`CdqLp+!IcS|k-Dkh(cOd>EwX)z(8-pMXd#69UFc*ER4{eDV!IQ8%go|;Wc2Pun2%_K^`@UEw;F6i1T$vLFUJ2|xfc9h3^4Q<<eLMDtNtxR;kM2fj^e{R9%qm5z`B^4l<5mz)F}RPOY37le)RYA0YGo5L7jBQ<cizea0?^}T@S0aL3(ITW>|x`bFKO;BjkSHRT@mGF?QEgguWk^FV3&!WpXS@e;}M0Gnu!XfH7?-icJDYuN^-qcNti4&YFVAE9e9zIoj)f=t34>Gu6*Hi7{6-=iz3SpFaHheILG`}eK+s!s*yRZWMDdg;W%E5eRC#;3)XN03kRwI^aVm{ORbFtzT@sGpkU6m*f1#BW9Qu)Hp)ye^PsWST;-cpZZP4MmiN)~bR_zw+MsBEAH1?=VWne9ec1E@e-($FQmwsbn)EHRjz?91%2FJ9V*^EexC!v&wpf-r^(1NX?UYUH{cq-l^HWP*saFt7>RUE%_7?eLvlk|Q3IZXn`uRumzvcTqjt5@u(-+@?5bt(u?i0YeV%=Jkig5aqPPBY=pWClm(r}X4+Fzir@Lj~C2hpDF;?1P!5GL<EdN^nbXt$Tq+z*_OHr1Vd&N{aaseM#Q^*mC!-mL?upIYG;(aO7QHh4C3b}?z~cpa~Ly5q9B$)ti>9lAsFf;FA-{wCZyU2V4&XqmqI)n}fvJEh+<#R-dHU1---0FrWt14Oik5>wG11SOpWmHeDpAqv5IwWpvxtIvK)>hiGYSbLR8^90*gVf#Thi9j(0o+-U0Y(F*2k|RkNQy;$vJKe-MoywN;79e*RgMZ*z3rvRVxK_P1^uRB6-H{LqEo^(fHL@wP(vP3WXS4PWy%;ld&EHIi<LfS{Tvc#*>KMzG-&lOZWQQtTH;?9#*t2bOH*&tvwBp+@^h+{<hh^v2RH4q@_o$eIS0sC<c~uiD-!HgngWp+DW--ngaam9UCmn7wgYEshK5RCXi#n<B9V_3JYm%Ws&(kJh31|#QZ|~{FtUr5Qj~y8JB&(xAQQr^x{>5CTOc3X*B)`>PGms+C|DF|Y(xNCkkQd65xZPT3Bo%;Trl50BCV=gq_n%(KzSG)J>2W1tEvfcc9aLFkI&MG0v>mE>GfogljGU3HjX-H0GWCRBYM_j=o8&5scH`b2Jkq6Y^E6~gG`HT(hdlfB!o_SVPnLthI0c=!Oc#&tfgZNc&FZP*1S9Ou$)y9R?t8=*y32^8fmOK=jpE^W5M++DK|r=d9&8WHgXZ)O(McR#*Bt(u*IzdEI?QAB1$22kP`J1IxUafkbB&Y3dFO2ur`p+?c8A3UoVWC*Ay#X*M<?rF4kLB^9Iwm(I2-VAbr8X+Hym!XKYKwli&_t_jU#0{1Kervg-p@A!%lk>Nw@Rzq6Uv(-u-mL+rZc2t-lf`?Wh;J{HNV`R{Nc(B+pS^2cg%>2cD~XExY5hv;3@KUb|gRprp%jt*|<6PahhbF-+;yb=6CaJ=28I6}<MN4WL<bq5HGPTPWTGLufTdSM_y?Qkh~P*<o5cGCph#HZ9H%{@j+V()3Q%!G`I6Hht&P(a%q#s>9g|<c5`1oG1Hp&^p0%ykt1Rdx2X_d&02WJ~?wG9t85<{i+OdDMgi93F3cCt2j|9#XoO^cK+lUKbo*D%^J?5@yNM^-LVK45Q%&`&-Oeyby;nFDnAhp0Kd^oWE<`Kd86$B_zrKU8<$5Kj~ibboJvG5#o`~#J_65%Ph->w10_1DB+#);_TqMY1;Pblo;5b|8)u`d)`Av^4QFDdc{9D@+e!8nMNE=M0279_YZlq2a#;4xCjA&bOCfa4g(MLb^m3=G-DKZ5+hT+h)^ihG*4ET&8419{m8Q5=3W1WW)3BD8+&eU|dHNSIf7Oc!hN(fh5Bi>#evm`R=`McXLeylO?0YWO$u4aJOb<zY2;2@S*sDhm>v{hk%-5~luw&Q!MRs+XodM?_ICW2HQ=*Rm6!umZzixA3OXO60^l++b07mdDBjU($?$1e=yi(H9NI79NX<|F=+rAxowQYOCDYXrWobw|)9n2~=^yFXn^Zr$QKGA-{<?HnDI33yHaZ<Ck*fFf2_XRQJeZO_Y-mc_u>lM-tn{DM0-Tqvk?c?}#P}%U?KwIvov+biFtm#!9%SP{`(d_|bEr0I3`BYtArjLMsg`0A9)Th*%Y7n<k9cp=Fix^R<j##Aw;52uwCdD0|U+p^9nZ1qwW@l<o;>d!FOO@&9ux`)U7r|{`sXQ3uug#U>tD`5|y+Zu8PN<LQWBHbKn7fPI<ke@5n@+}PKi)QesDRfG!fhZZMpv>k=9}rAD-_GmbfocBhCB{a?eg#&&qJmAB6A7Vvyr!>T1TCD4{wc$?x0V$U27-MW}LNjYRsgSa8+IP_xnXdnh)zYJmJI%Rh>mU#|XElQ_V!;i1oBh1yUA9#0=d>xm0sp8)S}pXZfC=@T*9DM!|SeuDps8p4GuYZzVU@o4G&V%pd|?`q&j5Glbb!<{eRPI_Uu^!FVUQZ<6bAAP--&H9O0ndSZ!s<sq=ngcSzSY(dUf<*rk2j!{Yy8^MqWu<_crOw73N%P7gPxh9UeZE#D^WKJA|Q-gCLp;!EY_VMOLd=7YCXP|u@WA~@L_vjd)TAVa|{E>I?y-3>WV&+drvjchTlzoc7d8BJxpGPl803hBio@BJHne*Tfd^GB5S4O5$t3Q_r@YD2qg|a-W!&7Lr-#m^}5P+J+P{uDz|Jz#&y6HK-)Qz4grtXF{fT=RxCR;ry7!73+&n8py&ANy48w_!3LzM8(90gy1gBj|simdpZ-J6~f*SiB<|Jl5fcRK>A&0lQV_eSg8yxz~G_+#qvTp1$NX!n3$A46{s)cli$TjwQVkJJYA)h34Fi)hIdOxX8nJc^KkeR9iH{m=v(N>xvG(%s19ue9lic0Dh(z+%*zx7?KvvE_jy^!Qgm>BFtz44GmrEPQe`8O3GmqHs(8n!LEA3M<cgvObh{vb3$0odSh;iHoP_aVK)Om(i43Tk{wlUiO=Lyq*~45qsPIYz#ZrV>hN{y;D^$pbDp?RBP+U*Q7t(-HXw7SJuyYc(pF?Ue4qLs;M4*b-PFZY)$s>daFZp+V-e9^N%~h1(b*G3SS~;x#rsU#8c6L*k!$^HM}dqm%h9;@tKY`QzzA1)G~FcJ#s1&(}_yZVea>K9IT=Jg}<ID<vDDt&5i}Eu)K4dB7R&NE;A|hJ5!SsU+2Z32_X%Z3@A@rLhs9BwFF#Z5_*2~t(0yz!=IuTesIgTPm<N?1Kn&?ro#R)hi~>X7O>0KAHV8Cb3SqJ{m)3NRr}39L-efWwVcf8Y!bl$LvJhFyOY$y5Z|>88M<^TOZoVoE=l~BvJ_UK);!R(cuRe*xay0-d-rbx5CrZ6_I<GuDdz^BSxOW2pbM9?qgKalRTk}8p)JSqnQAJdGPN825LXj^m#Jg5V(AmRLvc=a9d#hC%2*xj1@tzeF-=JM5JrY6EZ5og+vT!q>64*YiH6f{J8FtAQ@zgTvopv9`$nzdjcc80k%yI|TccZN#eJQ?Ftl)fGm}7sm@V(&$#Br|vile<asXRal2e8c{M#u6X9pp6vCldz#reW2>WdntPq!`|fB3I}GUpm}aC7xst{o^4t_@d1la9(FL_GUT`Of#5DquUW8iajjC0CSUTod|zYWeJfq4Sw@v+Ucc(VvR!e(s0OJvcZDytCri(ECNyKJ_~f+}{tiiYXGyv1%tHAKEYOQdgl-R6{LKSz+fFE~1y`a<t64-9k!XBQjG;vw~-~q01ZEQ`E&huSbb?n!FpnTn8(MwK<@NYvsHXw!{P{2Jv<dvn}S@qX+CLNeW=W^NYhv2<;$}XS}W!M1|t6m9~r4ULUU5pX#^$blw{n@%_`$VnQhkHU2h%I{WnvrQ7^M7Q7FogvGa#BWouO3#ESq#Vg)N#Y%;LqxN!fn&ZbNaI1UeD7-M)Z1XaT%Q*J=8g>9w&nzeX<Hf?%_Q+Fb1A4WNY9(0S(|TG47u=vDJGT+)=s<n>9IiWAxyLE+qRAYx%J+1BO%h}MJ}lZbClnZJzD)GPD(<({k8vT90D9$ecnM48D`q2Spg*XWPaB|lNrl~`oOyGH`79ZC)m6I<Bdcl95zFiMRPA<*3KwrmQ1C7#Gur938US+aN}S7r9kQ_;#2O`cyXo$uaWk-JRQIwX?}UwV&=Fw0Kk)}`Q*uJ<8QZgKzGkW440_T|9hZm+eg-_dYl$pN!sxCanLcT7&BSJ|+OqZ9?yIZz<}I~6ZuhXI2MNFUE`RTy{o(mELw+kxL*F=+@5;=A?_z??=*qcWZK3<QP9hOXj-G+rShTmz<!tu8>iyaIv90=6HSJLj9R$CYDl7J>DZ$%82#`PUu<D$L$cSZqQ3v|ZGqe22qrn|(oTk}(DOJ7C5NgmgDMho%&S>eJZWMa8s~>U+7Ohd~(Yu`*X8Qr51xLpl1}Pu+1K`Bhdqz%JHE2`U`Q<TnspN3`^ws*lR?(=Na7x=v25VN|Ka>Ll<1-ScSzEZjcFSHXx6uVx7RKY%if}Vf@y_|QCm&1JUS@28?mB6`G~IQa(M-sPbARbn_kHW~7<LJoB}_%yb5}wb!)ZRb9y+q0`(+pGS$c}H1s?0=YwmSMr*SmEnv0B>e%^Fv+_+z#gV}JrYHz!P7Fpfap7vtrxcv{=oo$&bnmVC&TnM-Tp$r*}XKJ{=mcF{$ZAXu2d%5BLtd~g7g^djdJEwOqWXMf-B@;jHvGeW=1;7iYDMB~gX>RN!b47Rj5-%>M?MTLHdES9u<$J;)$PXNj9mrPGH0LHLCJdBmT!AIH+R^dZ@!qK`uh;v6)s)jTOh@-?(8%ON-oExxSzOU$RTv<%i{vAXv}^*s6QlI6?51acvnH4U4k?9?<2@rupE+O~PV3zT{&1YO79&rXO2@^KL=@+Eb;$A=L{@hOZk3SPlB>nJ{+va|n3fLpWo3D0!tTQD?VZhV(>heqhx^BVqCB-@;8XjNIi5<B(!jtiA_tyAyUWe-z_SJRYS-~IOjh@Xh_BBpq<nzg8mb!hnAWhFyVA&)*;}`-^B+LgSAo;3O!RxN@l7pop6|S6?mvc`$-5Mk!dGLlqds@!=REg(TU2n0QJ}|7N0yJ$b6a-B>uh7af_JtK<ilamui)zO9Pp#ecnc>iQwgFKPCs$nE^fwlPnzht7@6NuYuzFR>bW|@_p1Wy%4;|A9z5^O5B}>ilrlm%OzXoFDU?Rb*&e;p=kD0_6`cQ~+Nx?4U{xF49?kUqnh!-Wp}^L&=br=}8hAPT1^e66fOH(c@PuQl{FD&Bsa8(PVa~V6SLCZ_xfIyX*I;-65WKLu?()S8Mi?bw*8|c5pA41L`l7>ZHQ6qmKDt=d#d69|<44C`ja2SlRRw6qKYKsFJLQ+<Q$c>d@SoTs13$W3Taa<4pFLw)61&+<xlO1Fq?*23tocx@qzm+mkEA{N+0=SA9I)s&-Dyybt8j)a>6nT--?bIK3$l4D))d!k*NHA2*p7sf<LDZ<5hAD#xX!YoQ1d!J8_bP7>@kB*@2tK=fM_Ajy-eie&v+?4?jhId){&_$EtLI)G)*A$N%*1+EOEviD%<yU9=(byJxJl=;EcdA%_zNVRSrgn&^(%Q)Pg)9Z1b}+Tt5!mzm^&2O!*k>4EDMxJIdpx1=ahlGz1zLd?BLerZ1{`jYfLrCQjzVrXTo&9>I>9jY#;ej~{uOMQoM}6!zAxb8b4A3ng|h7)vxC+|X(>Y#O_dp5zk&*|=qDP^%5P7y&VcCWiUtUJC0!75K?hO}dGaj*yF-_F(DHe@0@K?1-R@*iSK;oR%nV4NC1jRZ`!p=xk1(oyU~8&cXJo8+K(A4$37*e2R5isxB(hwq^VG_1Sxj&YfNLmNCLaSRP`vU?2;#o`WtYrq-=}1Hm@ilwW~-dr(Gr7B$|OKciwvY~kh`Md$5)`_@5@Ny+GbzKt8{YKyNj*@2;}oD>MRo=&^B^R#;gradhw-~ZhAH<AYS1h7kMjk%{o`})}rJ>-P-gqa}SPIDqz%Uz&98zW4BIkt>Jku8Z&SeIeExFMnf5}zIYY;>7b(gEE5Tn60r+TJ0T@kd*=E>ueIo5iR!Ogd<P{Z8X-eMwQnB-Smc@*0bIRQL<>F~fe57n=5ImwY9ykeD0L*pWWU6wUE1HF>Y7k%w;%Cy;pHlj*r_`u%{97uWT@mMZl2-K@wN2JKHf)9gO!p?l%h`?m8nwJrv?2jNFYZ7ll?^z<-=U(04Tt|HP*cw)%H=#y<rtCh59-iTJ+>%^FrhYk16aG4iRabP|{Ti-xLyT+~BB1Ra1!7eoODtPF=XpcP{8wYTHxs#1g*lK7)XsNlibF_UX)vV9nky5d0)Y!euuJ3mNy%QjH@JI3CIuAh!bVJAK^ee|{bB1Nz=Y)tZK0ZPF0!!5N9rCl|RN6sRXqOM1<AS4TbzUJyzv=K+IU}?8Vm%z<DEcaw$60@op4Q|tBDYMgk7`~F;dBSQUKa_RN**g*hG~F&5oi|zfCOlrrt~Y$<7O?6SX3&}W9qgBY`V4v)wz17yu}v85OO}X_Y2_2Pu9bo`|8u%8Spk9MK7>;7jol=3ro;At4FGoSD(;rY#d5UMN%6OG0mUH!Dv7Is@Gh10QvOgqSb3CEoZNEHg~SKcsaz1c2~mb#{h0mW1cFHKz;q2zunen!>8lXx}GZ%6Fs`pv!=-Tx#*1EOCjxjy7vN*v>xuxsk4yQ!_9aUY!y7O2&)|rS~U?v*OLti9tUgjIxIFT&xQs0_!9UUQuHCSG5rZ@i`#dXajuA`S8>&#hu2G$ET*Px;q0rn1Y3*v0LXGc4mUJCciZYQVCtB%d{kRKa%goUf}7l`?9*em3LOtPt90a5R}<~HpUfwnTQM1A>-y(Y=r#Px#rdHcO~%6;r0|Eya(qJ~;_0XR;l)k8^YsHl;HA`Y(JdRE$lC~WARq3S#15*Al=0pgo09x7Ci+X;5Stb~nI!YwlH>O8Mq@`V<JNkxNJoy^18zNh$asCL!A^k2st=3eF&ge?x+~dr!!Jog=8Zb73E||ZH-TK>U|-iE3!{EcX~QSY>^6`357-?jR^?kq%gv!d?E}Hb_&M6gyKZl9C#~gjt$0qVI$6gXuX>c(iLu3_w(3}$gVJWmua*-}@YzkCamo~Rpgxnecsq^L&&N8ILP?(`M!7ZH-hlIUW)$7~)mM{1F=#pmWMZ{B*m}?@t+*Z}EmAFoV0;g?6JnDTuUh1+ciCs`ym_@gS}XE-G>y#HxhShiMUtnNx;>gs*AL2!goU3)dLC~4Q*gB1_ni$^QsK`^`Nh>oA*wI?;%CX0&&PA`cDH!(ybu4x49$#2YVjNNm&zWrOV@y*1M52>%anV*=zSTl-EW|GX={98O8etryyK0lGg)8X)^h3AEhCGTELqMH7^d5mF^|<t>}kDvV4rS#APzs4+N_<c;Q`Op%F^rbyMFvUx><Dsl{*KA_nwn;Jw(s%S|IYN(yuYwAa!vySF7U<See&{_gO3G=T+LHIvsfqmsgp^zK@HTU7PfXRQ<GQ0-|GmwXYCc(yrEngwH+DINCewd$D!v&CNe&@}!+u6xt{bU*+ubr3~epf}(va<J%K(Pd<w@hAus!x1zLV-+5fdSYcf3`ll&BeXk^-TKmTK)nni68~Y2j+8$~J4uS1G@tEJrpV51c1KOe--nu0=CFnvB*pv$kxMvT8V{<mu6nw}Q&&O?ACnoapLg}s2uprJ1V{%<Ck+*h*b|F2pk*!T_wc6!IRZ6uTGZ<xk6bq!znkY|BPOCT-z;lyXjsRURm;DoA0K@JRCiEItq`C8!)(56^4+rVf1xkbc^|mc@`f)PB1baOef_0Oz<!%{+va8~GEcR)|NHS9BQc1m?ZUbIsxGu%mN4+R<@)3Wd+IqQ=0n1V=ey!xT5ZzOe@t*27+j#N)c9(5p^8%ruA6~~taN9fJLFGPl58xh)-&Ljx-<!tvWI9?^WKi^=_SjK%o+H-P@j|EodAPbPk5{{9ch(-<rvP@XA^qMX66+9EMU`gKsZ|MVv%~U93pp)|b>$tV<6{Z*I_)g?#Su<z!Hg@v3Oi{+U`~447uw#QOXW3C%=?(HXW4qZgnRMUJqRR@EgH7jeLr^>CxNc_eY#qxTB9n~=Q`Fla#X*k;&nHcez?biF5imMmB&HhgeO>x`^dE@m)|cJ|3z{U{{*>kRgef+Ihl56pv8S|diBvgihKJ)AR;<Ct`oc7+ytN5p7C!>$Cw$%hr<gC@0Hvh_Na$jLz5VO=|R~Pm&0TbgysFFDo+UPOmL;|7p{8Y32!)5!`&W~n8VHp3${M2ltqXW9<-B0W3NRle$~FZ70O*n&(U}j(F5KjS;etNH^dm%YQ5IjkC07)sa_2{lwLs??yk}L`|xAE5>nD{_W#dM^*{bYznJ&rAJ{)U{bp+3PiUXCzj>9{UqNJ=UXc9tKmM_aQuCkx_&*HoY{!XyKbb!X@?&38Jv#sBzhd7?{`!}65Bq-hA3w?eZ2t2fUAX_h{zc~5_n!d%SB{?l;WV57!m|_8i~khxm$VUag`h=o|EC6yUanVv6YO&F=c}yO%Rk>KEpFH)tJ31%T!zLse+Uv6#r5A5J)8aYM~}!)6H_yunq^0EqQzG~_|^0JYmbX7P1>wQ`j160iuTYx-21QB2wHu(-|GB*$pQZRlEa?@C2qamFf6S}%$onNJpb(r27mIfB(0GmjdTB+>A$;8@waKX$kC#tak#jkH~(f$gt%Rij3)8xe>VqDZ!{90(*JI9R#ox;7uyMZLI0fiZzTTwtDF8fNYj(QeT4phc<S?~wAhb+t~OowPoMfLO#Q_Cqtf4mX6VW3`lqkK=s*9_ZvN>ci5~pv<bRF-F16Eydq@}lzf`RL$NvThhQ8_"""

raw = zlib.decompress(base64.b85decode(AGENT_B85.strip()))
actual = hashlib.sha256(raw).hexdigest()
assert actual == EXPECTED_SHA256, f"agent bytes changed: {actual}"

out = pathlib.Path("/kaggle/working") if pathlib.Path("/kaggle/working").is_dir() else pathlib.Path(".")
(out / "main.py").write_bytes(raw)

print(f"wrote main.py  {len(raw):,} bytes  sha256 {actual[:16]}")
print("submit that file directly, or tar it if you prefer.")

## Checking it loads

Nothing clever here, just proof the file imports standalone and returns a legal action, which is the failure mode worth catching before a submission burns a slot.

In [ ]:
import importlib.util, pathlib, sys

path = pathlib.Path("/kaggle/working/main.py")
if not path.exists():
    path = pathlib.Path("main.py")

spec = importlib.util.spec_from_file_location("agent_check", path)
mod = importlib.util.module_from_spec(spec)
spec.loader.exec_module(mod)

blank = [[None] * 10 for _ in range(10)]
obs = {
    "day": 0, "hour": 0, "player": 0, "step": 0,
    "farms": [{"money": 3000, "farmer": [4, 4], "hands": [], "hires_today": 0,
               "tiles": blank, "unlocked_quadrants": ["NW"]} for _ in range(2)],
    "private": {"shed": {}, "seeds": {}, "inventories": [{}]},
    "market": {"prices": {}, "inventory": {}},
    "town": {"unlocked_shops": []},
}
action = mod.agent(obs, {"episodeSteps": 720})
print("first action:", action)
assert set(action) == {"farmer", "hands", "market"}
print("shape OK")

## The part I would take, if I were reading this

Not the schedules. Those are public and everyone has them.

The two checks that changed my results:

**Ask what fraction of your gate you already win.** Mine was 58/60 and it returned "no difference" to four separate real improvements over three days.

**Hold out the towns you select on.** Four of my routing candidates ranked first on the towns I picked them with and then came in *below* doing nothing on fresh towns. In one case first-on-selection was 13/14 and 23/41 held out, against 26/41 for no route at all. A selection set of ten or fifteen towns ranks roughly; it does not choose.

If you find a base schedule that beats 86/90 on a gate like this, I would genuinely like to see it — seven in a row hitting the same number is the kind of result that usually means the ceiling is somewhere else, and I have not found where.